---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-20: A Deep Dive into Transformer Architecture - II</h1>

# Learning agenda of this notebook
1. Recap of Previous Session
2. Why we need **Self-Attention**?
3. What is Self-Attention?
4. How Does Self-Attention Work?
5. What is **Masked (Causal) Self-Attention**?
6. What is **Cross-Attention**?
7. What is **Multi-Headed** Self-Attention?
8. **Layer Normalization**, **Residual Connections** and **Feed Forward Neural Network**
9. **LM Head:** From 6th Decoder Output to Token Prediction
10. Visualization of Transformer Architecture
11. Example Questions to Check your Understanding about Transformer Architecture
12. **KV Cache:**
    - **Why?**
        - a. The Problem: Generation Repeats Work
    - **What?**
        - b. The Idea: Remember the Keys and Values
        - c. Why Caching Is Allowed
        - d. What Exactly Is Stored (inference only, decoder only)
    - **How?**
        - e. How It Works: Prefill and Decode
        - f. KV-cache vs. no-KV-cache text generation with GPT-2
    - **The price**
        - g. The Price Tag: Memory
        - h. Making the Cache Smaller
    - Questions to Check your Understanding about KV Cache
13. **Mixture of Experts (MoE):**
    - a. The Idea: Many Experts, Few Used per Token
    - b. Where the Experts Live
    - c. The Router: Noisy Top-k Gating
    - d. Total vs Active Parameters
    - e. Build It: an MoE Layer in NumPy
    - f. Load Balancing: Keep Every Expert Busy
    - g. MoE in Practice: Batches, GPUs and Speed
    - h. MoE Models You May Know
    - Questions to Check your Understanding about MoE

# <span style='background :lightgreen' >1. Recap</span>

<h2 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The <b>Transformer architecture</b> was introduced in 2017 by eight researchers at Google (Vaswani et al.) in the paper <b>Attention Is All You Need</b>, presented at NeurIPS (Neural Information Processing Systems) 2017.</div></h2>



<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The <b>Transformer Architecture</b> has gone beyond NLP and text-based tasks: there exist vision-based Transformer architectures that have done really impressive tasks in Computer Vision, and audio-based Transformer architectures as well.</div></h3>

<div style="text-align:center;">
    <img src="../images/tfmr-007.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>



# <span style='background :lightgreen' >2. Why we need Self-Attention?</span>

**Example 1:** Consider the following two sentences, that demonstrate **Co-reference Resolution** (Working out which earlier word or entity a later expression (like a pronoun) refers to, such as deciding whether “it” means “animal” or “street.”

Word sense disambiguation: Picking the cor):
```
“The animal didn't cross the street because it was too tired."
“The animal didn't cross the street because it was too wide."
```
- How does a model know that “it” refers to  the word "animal" in the first sentence and to the word "street" in the second sentence?
    - In the first sentence, "it" refers to "animal", because animals get tired — streets do not get tired
    - In the second sentence, "it" refers to "street", because streets can be wide — animals are not described as wide in this context


**Example 2:** Consider the following two sentences, that demonstrate **Word Sense Disambiguation** (Picking the correct meaning of a word with several senses from its surrounding context, such as deciding whether “bank” means a financial institution or a riverside.):
```
“Money bank grows."
“River bank flows."
```
- How does a model know that “bank” refers to  a financial institute in the first sentence and refers to the land alongside a water channel in the second sentence?
    - In the first sentence, "bank" refers to a financial institution, because the word "money" is mostly used in financial context
    - In the second sentence, "bank" refers to a river bank, because the word "river" is mostly used in geographical context

**Example 3:** Consider the following two sentences, having word sense disambiguation, coreference resolution and case sensitivity as a bonus issue:
```
“She ate an apple after lunch because it was fresh and sweet."
“She bought an apple because her old phone stopped working."
```
- How does a model know that “apple” refers to  fruit in the first sentence and refers to company in the second sentence?
    - In the first sentence, "apple" refers to a fruit, because of the contextual clues like "ate", "lunch", "fresh" and "sweet"
    - In the second sentence, "apple" refers to a company, because of the contextual clues like "bought", "phone" and "stopped working"
    - The word "apple" in the second sentence is in lower case, but, the contextual clues alone would correctly disambiguate it to company and not fruit

# <span style='background :lightgreen' >3. What is Self-Attention?</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Self-attention is a mechanism that allows a token to look at all other tokens in the sequence and decide, <b>"who to listen to"</b> and <b>"how much to listen to"</b> and then combine that information.</div></h3>


<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The output of the self-attention block is a contextual representation of each token, which captures relationships between words beyond their standalone meanings and positional information.</div></h3>

<div style="text-align:center;">
    <img src="../images/self-attention-007.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

<br><br><br><br>
# <span style='background :lightgreen' >4. How Does Self-Attention Work?</span>

<h1 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Self-attention lets each token decide “who to listen to,” how much to listen, and then combine that information.</div></h1>

<div style="text-align:center;">
    <img src="../images/self-attention-013.png"
         style="max-width:800px; width:100%; height:auto; display:inline-block;">
</div>

<img align=right src="../images/self-attention-14.png" width="600">

<h1 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Apple released a new iPhone model last week</div></h1>

<br><br><br><br><br>

<br><br><br><br>
<img align=right src="../images/self-attention-008.png" width="700">

# Step 1: Generate Q, K, and V Matrices
- The very first thing self-attention does is take each token's 512-dimensional input vector and project it into three completely different spaces; creating three new vectors called the Query vector ($\vec{q}$), the Key vector ($\vec{k}$) and the Value vector ($\vec{v}$)
- To do this we multiply the input matrix `X` by three separate learned weight matrices: `W_Q`, `W_K` and `W_V`,  and the results are three matrices: **`Q`**, **`K`** and **`V`**, each of shape **8 × 512**  (one row per token)
- Now what do these three vectors actually mean? Think of it like a search engine:
    - The **Query vector $\vec{q}$** is actually a question, asked by a token to all tokens including itself, about the information it needs from other tokens to understand itself in context. You can think of it as a Google search query.
    - The **Key vector $\vec{k}$** of a token is a response from a token, telling what information it can provide to other tokens. You can think of it as a web page title to the Google search query.
    - The **Value vector $\vec{v}$** of a token contains the real information. You can think of it as the web page content in response to the Google search query.
- These three weight matrices `W_Q`, `W_K` and `W_V` are learned during training — the model gradually learns the best way to project input vectors into meaningful query, key and value spaces — so that the attention mechanism learns to focus on the right relationships for the task at hand
- This entire step happens as one efficient matrix multiplication — all 8 tokens generate their Q, K and V vectors simultaneously in parallel

```
Query projection:         Q = X × W_q            (8 × 512) = (8 × 512) × (512 × 512)
Key projection:           K = X × W_k            (8 × 512) = (8 × 512) × (512 × 512)
Value projection:         V = X × W_v            (8 × 512) = (8 × 512) × (512 × 512)
```
- Where:
    - `X`: Input embeddings with positional encoding, shape `(seq_len, d_model)`. In our example it is `(8,512)`
    - `W_q, W_k, W_v`: Learnable weight matrices, each of shape `(d_model, d_k)`. In our example it is `(512,512)`
    - `d_k`: Dimension of queries/keys (typically d_model / num_heads)
        - Where num_heads is the number  of heads in the Multi-head attention 
- Finally we have three matrices **Q**, **K** and **V** each containing eight stacked $\vec{q}$, $\vec{k}$ and $\vec{v}$ vectors respectively, each corresponding to one input token.
- If the input embedding size is 512 dimension, then the size of **Q**, **K** and **V** matrices is 8 x 512. (where 8 is the count of input tokens in sequence)

<br><br><br><br><br><br><br><br><br><br>
<img align=right src="../images/self-attention-009.png" width="900">

# Step 2: (Generate Similarity Matrix)

- Now that we have our **Q** and **K** matrices, we compute a **similarity score** between every pair of tokens, that tells us how relevant each token is to every other token.
- For example, to calculate the similarity of a vector with all other vectors we take the **$\vec{q}$** vector of token **"apple"** and take its dot product with **$\vec{k}$** vectors of all 8 tokens (including itself) to get 8 similarity scores.
- This entire step happens as one efficient matrix multiplication, i.e., we multiply the Query matrix by the **transpose** of the Key matrix:
```
Scores  =  Q  ×  Kᵀ       shape: (8 × 8)
```
- The result is an **8 × 8 matrix**, where every row corresponds to one token, and every column corresponds to one token, and the value at position (i, j) tells us how similar **token i's query** is to **token j's key** (i.e., how much token i should attend to token j)
- A large positive **dot product** means the query of token i and the key of token j point in a similar direction — token j has the kind of information token i is looking for, so token j is **relevant** to token i. (Relevance is measured in the learned query/key space; it is not the same as plain word similarity.)
- A dot product near zero means the two vectors are (almost) perpendicular — token j is not relevant to token i
- A negative dot product means the two vectors point in opposite directions. After Softmax (Step 4), both near-zero and negative scores receive small attention weights
- The score matrix is **not symmetric**: S[i][j] = q_i·k_j is generally different from S[j][i] = q_j·k_i, because queries and keys are produced by different weight matrices
- At this stage we have raw attention scores — but they have one problem — their values can be very large or very small depending on the dimensionality of the vectors — and this causes problem when we will take softmax later. One of the characteristic of softmax is that for very large and very small values, it maps them to probability close to 0.99 and 0.01.

<br><br><br><br><br> <br><br><br><br><br><br><br><br><br><br>
<img align=right src="../images/self-attention-010.png" width="600">

# Step 3: (Scale)
- The issue with the above similarity matrix is that each score is a dot product $q \cdot k = q_1k_1 + q_2k_2 + \dots + q_{d_k}k_{d_k}$, i.e., a sum of $d_k$ terms. If the components of q and k are independent with mean 0 and variance 1, this sum has mean 0 and **variance $d_k$** (standard deviation $\sqrt{d_k}$). With $d_k = 512$, typical scores are around ±22.6, i.e., very large numbers.
- Later we apply Softmax to each row of the score matrix. When its inputs are large, Softmax **saturates**: the largest score gets almost all the probability (close to 1.0) and all other tokens get almost 0 (a near one-hot row). In this region the gradients of Softmax are almost zero, so the model learns very slowly (vanishing gradients).
- Solution is we scale by dividing the above matrix having similarity scores by $\sqrt{d_k}$, the square root of the **key/query dimension** $d_k$. Dividing a quantity whose standard deviation is $\sqrt{d_k}$ by $\sqrt{d_k}$ brings its variance back to 1, whatever the value of $d_k$
- In this single-head walkthrough $d_k = d\_model = 512$, so here we divide by $\sqrt{512}$. In multi-headed self-attention $d_k = d\_model / num\_heads$ (e.g. $512/8 = 64$), and each head scales by $\sqrt{64}$
- Note it is always $\sqrt{d_k}$ and **never** $\sqrt{d\_model}$ — the two only coincide when there is a single head, as in this example

<br><br><br><br><br><br><br><br><br><br><br><br><br><br><br>
<img align=right src="../images/self-attention-011.png" width="900">

# Step 4: (Apply Softmax)
- Now we apply **Softmax row by row** to the scaled score matrix
- After Softmax — every row of our 8 × 8 matrix now contains 8 values/probabilities that are all positive and sum to exactly 1.0
- Each value tells us — **how much should this token attend to that token** — expressed as a percentage of attention
- These are the **attention weights** — and they are the heart of the self-attention mechanism — they encode the model's understanding of which tokens are most relevant to each other in this specific sentence
- Softmax takes exponent of each value and divides it with sum of all exponent across all values in that row using:

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j=1}^{n} e^{x_j}}$$

- Where:
    - $x_i$ = the score for position $i$ (one value from the scaled attention score row)
    - $e^{x_i}$ = Euler's number $e \approx 2.718$ raised to the power of $x_i$ — this ensures all values become positive
    - $\sum_{j=1}^{n} e^{x_j}$ = sum of all exponentiated values in that row; this normalises the values so they sum to exactly 1
    - $n$ = number of tokens in the sequence

<br><br><br><br><br><br><br><br><br><br><br><br><br><br><br>


# Step 5: (Compute Final Context-Aware Embeddings)

<img align=right src="../images/self-attention-012.png" width="1100">

- Now that we have the attention weights from Step 4, which is a matrix where every row sums to 1.0 and tells us exactly how much each token should attend to every other token.
- We use these weights to compute the final context-aware embedding for each token by multiplying the **attention weight matrix** with the **Value matrix V**:

$$\text{Output} = \text{Attention Weights} \times \mathbf{V}$$

$$\text{Output} = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right) \times \mathbf{V}$$

- The above multiplication actually takes a weighted sum of ALL value vectors $\vec{v}$ across the entire sequence
- The final output is a matrix of shape **(8 × 512)** — one 512-dimensional **context-aware embedding per token** (exactly the same shape as our input). Each vector now carries not just the meaning of its own token but a carefully weighted blend of information from every other token in the sequence.
- This is the entire point of self-attention — the output embeddings are no longer static lookup vectors — they are **dynamically constructed representations** that reflect what each token means specifically in the context of this particular sentence.
<br><br><br><br><br><br><br><br><br><br>

# <span style='background :lightgreen' >5. What is Masked (Causal) Self-Attention?</span>
- The masked self-attention sub-component that exists inside the decoder layer, is quite similar to the self-attention inside the encoder layer (just discussed).
- The simple self-attention is bi-directional and is allowed to attend to all positions in the input sequence simultaneously.
- The masked self-attention is uni-directional and is only allowed to attend to earlier positions in the output sequence **and to itself**.
- Auto-regressive Language  (Decoder-only) models predict **next token**, so to avoid cheating the models must not attend to future tokens

<h3 align="left"><div class="alert alert-success" color=magenta style="margin: 20px">Predict next word in the sentence: <b>"Ask not what your country can do for ......"</b></div></h3>

### Steps for Masked Self-Attention

**Masked self-attention** follows the exact same five steps with one single modification inserted between Step 3 and Step 4
```
Step 1 — Generate Q, K, V matrices          ← identical to simple self-attention  
Step 2 — Compute similarity scores  QKᵀ     ← identical to simple self-attention  
Step 3 — Scale by √d_k                      ← identical to simple self-attention  
           ↓
```
<span style="color:red; font-weight:bold;">ADD CAUSAL MASK</span>  
```
           ↓  
Step 4 — Apply Softmax row-wise             ← masked positions become exactly zero  
Step 5 — Multiply by V                      ← identical to simple self-attention
```
<br><br><br>
### Detailed Step-by-Step Masked Self-Attention

- **Step 1:** Generate Q, K, V by multiplying input X with learned weight matrices
- **Step 2:** Compute QKᵀ to get raw similarity scores between all token pairs
- **Step 3:** Divide by √d_k to scale the scores to a reasonable range

<div style="text-align:center;">
    <img src="../images/self-attention-15.png"
         style="max-width:1000px; width:100%; height:auto; display:inline-block;">
</div>

- **Step 4:** Apply Softmax row-wise to convert scores to attention weights summing to 1.0
    - In masked self-attention, we add a mask to the resulting matrix of step 3 before taking the softmax (which prevents the model from attending to future tokens)
    - **Why is the mask needed?** During **training**, the whole target sentence is fed to the decoder at once, so that all positions are trained in parallel (each position learns to predict the token that follows it). Without the mask, the position of "for" could simply look at the next token "you", which is the very answer it is supposed to predict. The mask hides those future tokens, so training matches the way the model is used at inference time, where it generates one token at a time and the future tokens do not exist yet
    - The mask has one more useful consequence: because no token can see the tokens to its right, appending a new token at the end **never changes** the Q, K and V vectors of the earlier tokens. This is what makes the **KV cache** possible
    - How masking works:
        - Apply a **strictly** upper-triangular mask with `−∞` (above the diagonal only; the diagonal stays 0, so each token can still attend to itself)
        - Add mask to attention scores
        - Softmax turns masked positions into zero
        - Result: Token *i* can attend only to tokens ≤ *i*
        - In code, a very large negative number (e.g., −1e9 or the most negative float) is often used instead of a true −∞; after Softmax the result is the same


<div style="text-align:center;">
    <img src="../images/self-attention-16.png"
         style="max-width:1200px; width:100%; height:auto; display:inline-block;">
</div>

>- The attention weights are rounded to 2 decimal places, so a row may add up to 0.99 or 1.01 instead of exactly 1.0


- **Step 5:** Compute Final Context-Aware Embeddings
  
<div style="text-align:center;">
    <img src="../images/self-attention-17.png"
         style="max-width:1000px; width:100%; height:auto; display:inline-block;">
</div>

<br><br><br><br><br><br><br><br>

# <span style='background :lightgreen' >6. What is Cross-Attention?</span>

<img align=right src="../images/self-attention-18.png" width="700">

- Cross-attention exists inside the decoder layers of encoder-decoder transformer architectures such as the original Transformer, BART and T5. (Decoder-only models such as GPT and LLaMA have no encoder, and therefore no cross-attention.)
- It is called *cross*-attention (and not *self*-attention) because the queries come from one sequence (the decoder) while the keys and values come from a different sequence (the encoder output).
- It acts like a bridge between the encoder and the decoder layers.
- The input to the cross-attention is output of masked self-attention, which is a rich representation of what has been generated so far.
- The decoder however, has not yet looked at the input sentence at all, which is required for tasks like translation and summarization.
- The cross-attention component fixes this:
    - The **Query comes from the decoder** asking the question "what information do I need from the input sentence right now?"
    - The **Key comes from the encoder** representing what each input token has to offer
    - The **Value comes from the encoder** containing rich contextual information from the encoder output

**Cross-attention** follows the exact same five steps with one single modification, i.e., K and V come from the encoder
```
Step 1 — Generate Q, K, V:                  ← THE ONLY DIFFERENT STEP
           Q  =  decoder_output  ×  W_Q     (from decoder)
           K  =  encoder_output  ×  W_K     (from encoder)
           V  =  encoder_output  ×  W_V     (from encoder)

Step 2 — Compute similarity scores:
           Scores  =  Q  ×  Kᵀ             ← identical to before

Step 3 — Scale:
           Scaled  =  Scores / √d_k        ← identical to before

Step 4 — Apply Softmax:
           Weights  =  softmax(Scaled)     ← NO mask added — full attention
                                              to all encoder positions

Step 5 — Combine with V:
           Output  =  Weights  ×  V        ← identical to before
```
- The encoder output does not change while the decoder is generating, so the K and V of cross-attention are computed **only once** per input sentence and reused at every decoding step (see Q8(f) and §12).


<br><br><br><br><br><br>

# <span style='background :lightgreen' >7. What is Multi-Headed Self-Attention?</span>

<!-- ===================== ROW 1: Big picture + Figure 1 ===================== -->
<div style="display:flex; gap:28px; align-items:center; margin-bottom:40px;">

<div style="flex:0 0 40%;">

A single attention operation gives **one** set of attention weights, so it captures only **one** kind of relationship at a time. In *"The animal didn't cross the street because **it** was too tired"*, we want one part of the model to work out what *it* refers to, another to follow grammar, and another to look at nearby words.

**Multi-head attention** runs several small attention operations (**heads**) **in parallel**. Each head learns its own relationship, and the results are combined at the end.

#### Reading Figure 1

- The orange **Multi-Head Attention** blocks are where this happens in the Transformer.
- **Middle zoom-in:**
  - The 3 bottom **Linear** boxes are $W^Q, W^K, W^V$. They create Q, K, V from the input.
  - The stacked "shadows" mean there are **h** copies, one per head.
  - The purple box is attention, computed separately in each head.
  - **Concat** joins the head outputs side by side.
  - The **Linear** on top is the **output projection $W^O$**.
- **Right zoom-in:** what happens inside **one** head.

> ⚠️ The **Linear** at the very top of the decoder, before Softmax, is **not** $W^O$. It maps 512 dimensions → **vocabulary size** to predict the next token.

</div>

<div style="flex:1; text-align:center;">
<img src="../images/multi-headed-self-attn-1.png" style="width:100%; max-height:80vh; object-fit:contain;">
</div>

</div>

<!-- ===================== ROW 2: Step-by-step + Figure 2 ===================== -->
<div style="display:flex; gap:28px; align-items:center; margin-bottom:40px;">

<div style="flex:0 0 62%;">

#### Step by step (Figure 2, bottom → top), with $d_{\text{model}} = 512$, $h = 8$

**① Input** $X$ has shape $(\text{seq\_len} \times 512)$. In self-attention, Q, K and V are all computed from this same $X$.

**② Project, separately for each head:** each head $i$ has its own $W_i^Q, W_i^K, W_i^V$ of shape $(512 \times 64)$:
$$Q_i = XW_i^Q,\quad K_i = XW_i^K,\quad V_i = XW_i^V \quad (\text{seq\_len} \times 64)$$
Each head works in $d_k = 512/8 = 64$ dimensions.

**③ Attention in each head (yellow box):**
$$Z_i = \text{head}_i = \text{softmax}\!\left(\frac{Q_iK_i^T}{\sqrt{64}}\right)V_i \quad (\text{seq\_len} \times 64)$$

**④ Concat (green box):** place $Z_0 \dots Z_7$ side by side → $(\text{seq\_len} \times 512)$.

**⑤ Output projection $W^O$ (blue, $512 \times 512$):**
$$\boxed{M = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\,W^O} \quad (\text{seq\_len} \times 512)$$

**⑥ Add & Norm:** $M$ has the **same shape as $X$**, so it is added back to $X$ through the residual connection.

</div>

<div style="flex:1; text-align:center;">
<img src="../images/multi-headed-self-attn-2.png" style="width:100%; max-height:80vh; object-fit:contain;">
</div>

</div>

<!-- ===================== ROW 3: Why W^O + summary ===================== -->
<div style="display:flex; gap:28px; align-items:flex-start;">

<div style="flex:1;">

#### Why do we need $W^O$?

After Concat, the heads only **sit next to each other**: dims 0–63 come from head 0, dims 64–127 from head 1, and so on. No head has seen what the others found. $W^O$:

1. **Mixes information across heads.** Each output dimension can combine all 8 heads.
2. **Maps back to $d_{\text{model}}$,** so $M$ can be added to the residual stream.

> **Equivalent view:** split $W^O$ into 8 blocks $W^O_i$ of shape $(64 \times 512)$:
> $$M = \sum_{i=1}^{h} \text{head}_i\, W^O_i$$
> Each head **writes its own contribution**, and the contributions are **summed**.

In code: `o_proj` (LLaMA, Mistral, Qwen), `out_proj` (PyTorch), `c_proj` (GPT-2).

#### Why many small heads instead of one big one?
- **Same cost:** 8 heads × 64 dims ≈ 1 head × 512 dims.
- **More expressive:** 8 **different** attention patterns instead of 1.

</div>

<div style="flex:1;">

#### Shape summary

| Tensor | Shape |
|---|---|
| $X$ | (seq_len × 512) |
| $W_i^Q, W_i^K, W_i^V$ | (512 × 64), × 8 heads |
| $Q_i, K_i, V_i$ | (seq_len × 64) |
| $Q_iK_i^T$ | (seq_len × seq_len) |
| $Z_i = \text{head}_i$ | (seq_len × 64) |
| Concat | (seq_len × 512) |
| $W^O$ | (512 × 512) |
| $M$ | (seq_len × 512), same as $X$ |

> 🔑 **Keep this in mind:** every head produces its **own K and V**. In §12 we will see that storing K and V for every head is what makes the **KV cache** large, and that **MQA / GQA** shrink it by letting several query heads **share** K and V. The query heads, Concat width and $W^O$ stay the same.

</div>

</div>

# <span style='background :lightgreen' >8. Layer Normalization, Residual Connections and Feed Forward Neural Network</span>

<!-- ===================== ROW 1: Figure + how to read it ===================== -->
<div style="text-align:center;">
<img src="../images/dense-nn1.png" style="width:100%; max-height:65vh; object-fit:contain;">
</div>

<div style="display:flex; gap:28px; align-items:flex-start; margin-bottom:40px;">

<div style="flex:1;">

**Left: the full Transformer (Pre-LN version)**

Every sub-layer is preceded by a green **Norm**. Each encoder layer has **2** (before self-attention and before FFN). Each decoder layer has **3** (before masked self-attention, cross-attention and FFN). There is also one **final Norm** at the top of each stack.

</div>

<div style="flex:1;">

**Middle: one layer, zoomed in**

Data flows bottom → top: **Norm → Self-Attention → ⊕ → Norm → FFN → ⊕**. The **red arrows** are the **residual (skip) connections**. They carry the input around each sub-layer and add it back at the ⊕.

</div>

<div style="flex:1;">

**Right: inside the FFN**

Each of the **3 tokens** (512 dims) is expanded to **2048** dims by $W_1$ with ReLU, then compressed back to **512** by $W_2$. The **same** $W_1, W_2$ are applied to every token, independently of the others.

</div>

</div>

<!-- ===================== ROW 2: Layer Normalization ===================== -->
<div style="display:flex; gap:28px; align-items:flex-start; margin-bottom:40px;">

<div style="flex:0 0 52%;">

### ① Layer Normalization

**Goal:** Keep each token's vector in a stable range, preventing any single value from becoming too large or too small. This keeps the gradients stable during back-propagation, so values neither explode nor vanish as they pass through dozens of layers.

**How:** for **one token at a time**, shift and scale its 512 numbers so they have **mean 0 and variance 1**. Then let the model rescale them with two learned vectors, $\gamma$ and $\beta$:

$$\text{LayerNorm}(\mathbf{x}) = \boldsymbol{\gamma} \odot \underbrace{\frac{\mathbf{x} - \mu}{\sqrt{\sigma^2 + \epsilon}}}_{\hat{\mathbf{x}}:\ \text{mean 0, var 1}} + \boldsymbol{\beta}$$

$$\mu = \frac{1}{d}\sum_{i=1}^{d} x_i \qquad\qquad \sigma^2 = \frac{1}{d}\sum_{i=1}^{d} (x_i - \mu)^2$$

**Worked example** (a toy token with $d = 4$, ignoring $\epsilon$):

- $\mathbf{x} = [2,\ 4,\ 6,\ 8]$
- $\mu = 20/4 = 5$
- $\sigma^2 = (9+1+1+9)/4 = 5$, so $\sigma \approx 2.236$
- $\hat{\mathbf{x}} = [-1.34,\ -0.45,\ 0.45,\ 1.34]$ (mean 0, variance 1)
- With $\gamma = 2,\ \beta = 0.5$: output $= [-2.18,\ -0.39,\ 1.39,\ 3.18]$

**Key points**

- $\mu$ and $\sigma^2$ are computed **per token, across its features**, and recomputed for every input, in training **and** inference.
- $\gamma$ and $\beta$ are **shared by all tokens**. Each LayerNorm has its own pair: 2 × 512 = **1024 parameters**.
- **Why $\gamma, \beta$?** Forcing mean 0 / variance 1 could limit what the model can represent. $\gamma, \beta$ let it learn the best scale and offset for each dimension, and even undo the normalization if that helps.
- **Unlike BatchNorm**, LayerNorm does not depend on the other examples in the batch, so it works with any batch size and any sequence length.

</div>

<div style="flex:1;">

#### What each symbol means

| Symbol | Meaning | Shape | Learned? |
|---|---|---|---|
| $\mathbf{x}$ | one token's vector (input to Norm) | (512) | — input |
| $d$ | number of features $= d_{\text{model}}$ | 512 | — fixed |
| $\mu$ | **mean** of this token's 512 values | scalar | ❌ computed |
| $\sigma^2$ | **variance** of this token's 512 values | scalar | ❌ computed |
| $\epsilon$ | tiny constant so we never divide by 0 (e.g. $10^{-5}$) | scalar | ❌ fixed |
| $\hat{\mathbf{x}}$ | normalized vector: mean 0, variance 1 | (512) | ❌ computed |
| $\boldsymbol{\gamma}$ | **scale** (gain), one per dimension, starts at **1** | (512) | ✅ learned |
| $\boldsymbol{\beta}$ | **shift** (bias), one per dimension, starts at **0** | (512) | ✅ learned |
| $\odot$ | element-wise multiplication | — | — |

#### RMSNorm (modern variant, used in LLaMA, Mistral, Qwen)

$$\text{RMSNorm}(\mathbf{x}) = \boldsymbol{\gamma} \odot \frac{\mathbf{x}}{\sqrt{\frac{1}{d}\sum_{i} x_i^2 + \epsilon}}$$

RMSNorm drops the mean subtraction ($\mu$) and the shift ($\beta$). It only rescales by the root-mean-square. It is cheaper to compute and works just as well in practice.

</div>

</div>

<!-- ===================== ROW 3: Residual + FFN ===================== -->
<div style="display:flex; gap:28px; align-items:flex-start; margin-bottom:40px;">

<div style="flex:1;">

### ② Residual (skip) connections

The **input of each sub-layer is added to its output** (the ⊕ circles and red arrows):

$$\text{output} = \mathbf{x} + \text{Sublayer}(\mathbf{x})$$

**This preserves the real information from fading away**

</div>

<div style="flex:1;">

### ③ Position-wise Feed Forward Network (FFN)

This is a two layer fully connected feed-forward neural network that exists at the end of each encoder and decoder layer. Applied to **every token separately and identically**, with the same weights for all positions:

$$\text{FFN}(\mathbf{x}) = \max(0,\ \mathbf{x}W_1 + b_1)\,W_2 + b_2$$

| Step | Operation | Shape per token |
|---|---|---|
| Input | $\mathbf{x}$ | 512 |
| ① Expand | $\mathbf{x}W_1 + b_1$, $W_1$: (512 × 2048) | 2048 |
| ② Activate | ReLU: $\max(0, \cdot)$ | 2048 |
| ③ Compress | $\cdot\, W_2 + b_2$, $W_2$: (2048 × 512) | 512 |

- The first transformation uses the weight matrix $W_1$ of shape (512 × 2048) to expand each token's representation from 512 dimensions to 2048 dimensions. This allows it to capture more complex patterns and interactions within each token's representation. This uses ReLU activation function that introduces non-linearity.
- The second  transformation uses weight matrix $W_2$ of shape (2048 × 512) to shrink each token's representation back from 2048 dimensions to 512 dimensions and uses no activation function.
- The factual knowldege in trnaformers is primarily stored in these FFN weight matrices.
- **Attention vs FFN:** attention **mixes information between tokens**, while the FFN **processes each token on its own**, refining what that token now "knows".
- Finally, each output vector of the FFN contains contextual information gathered by the attention mechanism, combined with stored factual knowledge learned by the FFN during training.
- **Parameters per layer:** FFN ≈ 512·2048 + 2048·512 ≈ **2.1M**, versus attention ≈ 4·512·512 ≈ **1.05M**. The FFN holds about **⅔ of each layer's parameters**.

</div>

</div>

<!-- ===================== ROW 4: Modern improvements ===================== -->

### ④ Later improvements used in modern LLMs

Since 2017, three pieces of the layer have been **swapped for better versions**. The overall structure stays the same: attention → FFN, each with a residual connection.

<div style="display:flex; gap:28px; align-items:flex-start;">

<div style="flex:1;">

#### ④-a RMSNorm
**Replaces:** LayerNorm

LayerNorm does two things: it **centers** the values around 0 and **rescales** them. It turned out that only the **rescaling** really matters.

RMSNorm **only rescales**. It is simpler and faster, with the same quality.

**Used by:** LLaMA, Mistral, Qwen, Gemma, DeepSeek

</div>

<div style="flex:1;">

#### ④-b SwiGLU
**Replaces:** ReLU in the FFN

ReLU is an **on/off switch**: a value either passes or becomes 0.

SwiGLU adds a **gate**, a second branch that learns **how much** of each value to keep, like a **volume knob** instead of an on/off switch.

The result is better quality for the same compute.

**Used by:** PaLM, LLaMA, Mistral, Qwen

</div>

<div style="flex:1;">

#### ④-c Mixture of Experts (MoE)
**Replaces:** the single FFN

Instead of one FFN, there are **many FFNs (experts)** plus a **router** that sends each token to only a **few** of them.

**Analogy:** a hospital with many specialists, where a receptionist sends each patient to the 2 best-suited doctors.

The model holds a huge amount of knowledge but stays fast. DeepSeek-V3, for example, has 671B parameters but uses only 37B per token. More in §13.

**Used by:** Mixtral, DeepSeek-V3, gpt-oss

</div>

</div>

<br><br><br><br><br><br>

# <span style='background :lightgreen' >9. From 6th Decoder Output to Token Prediction</span>

<h1 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Predict next word in the sentence: <b>"Ask not what your country can do for ......"</b></div></h1>

<img align=right src="../images/logits-prob.png" width="1000">

- Let us now trace exactly what happens inside the transformer after the input sentence has passed through all six decoder layers.
- Here we use a **decoder-only** (GPT-style) model with the sizes of the original paper: 6 decoder layers, d_model = 512, and a vocabulary of about 37,000 tokens (the shared English-German BPE vocabulary used in the paper).
- Our input sentence is the famous JFK quote — **"Ask not what your country can do for..."** — and the model's job is to predict the missing ninth word.
- In this example, the output of the last decoder layer will be eight vectors enriched with contextual information along with stored factual knowledge.
- We only take **the very last vector**, i.e.,  for the token **"for"**, which has already attended to all previous tokens through the masked self-attention mechanism. So its vector already contains everything the model knows about the entire sentence up to this point. It is the most information-rich summary of the full context and it is the only one we need to predict the next word.
- (This is at *inference* time. During *training*, the LM head is applied to **every** position, and each position is trained to predict the token that follows it, so the 9-token sentence "Ask not what your country can do for you" gives 8 training examples at once: each of the first 8 tokens predicts the token that follows it.)
- This single 512-dimensional vector now enters the **Language Model Head** — or LM Head for short. The LM Head is simply a single linear layer with no activation function: `nn.Linear(512, 37000)`. It takes our 512-dimensional vector living in **model space** and projects it into a 37,000-dimensional vector living in **vocabulary space** — one value per token in the entire vocabulary.
```
input:   (1 × 512)    ← model space
output:  (1 × 37000)  ← vocabulary space
```
- Each value in this output vector represents a raw score called logits, that can be any number from negative infinity to positive infinity. A high logit for a particular token means the model thinks that token is a likely next word. A low or negative logit means the model thinks it is unlikely.
- These raw logits are hard to interpret directly because they are unbounded. So we apply **Softmax** to convert them into proper probabilities
- After Softmax we now have 37,000 probability values where every single value is between 0 and 1 and all 37,000 values sum to exactly 1.0.
- Each value now tells us **"the probability that this vocabulary token is the correct next word?"**. Most of these 37,000 probabilities will be extremely small — close to zero. Only a handful of tokens will receive meaningful probability mass. The model has essentially distributed its confidence across the entire vocabulary.
- Now we simply find the index with the **highest probability** using argmax, which picks token ID corresponding to the word **"you"**.
- Always picking the most probable token is called **greedy decoding**. Chat models usually **sample** from the probability distribution instead, controlled by **temperature** (flattens or sharpens the distribution), **top-k** (sample only from the k most probable tokens) and **top-p** (sample from the smallest set of tokens whose probabilities add up to p). You can try these settings in the Transformer Explainer (§10).

# <span style='background :lightgreen' >10. Visualization of Transformer Architecture</span> 

## Visualizing Self-Attention:
#### Transformer Explainer: https://poloclub.github.io/transformer-explainer/
#### BertViz (interactive attention heads, with ready-to-run Colab notebooks): https://github.com/jessevig/bertviz
#### Online tool: https://huggingface.co/spaces/exbert-project/exbert
#### Embedding Projector (visualizes embeddings, not attention): https://projector.tensorflow.org/


<div style="text-align:center;">
    <img src="../images/tfmr-explainer.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>


<br><br><br><br>
# <span style='background :lightgreen' >11. Example Questions to Check your Understanding about Transformer Architecture</span>

## **Q1. Self-Attention by Hand**

You have a sequence of **3 tokens** with embedding dimension **d_model = 4**.
The input embedding matrix (with positional encoding already added) is:
```
X = [ [1.0,  0.0,  1.0,  0.0],    ← token 1
      [0.0,  1.0,  0.0,  1.0],    ← token 2
      [1.0,  1.0,  0.0,  0.0] ]   ← token 3

Shape of X: (3 × 4)
```

The weight matrices are (d_k = 4):
```
W_q = W_k = W_v = Identity matrix (4 × 4)
      [[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]]
```

- **(a)** Compute the Q, K, and V matrices.
          What is the shape of each matrix?

- **(b)** Compute the raw similarity matrix `S = Q × Kᵀ`.
          Show the full (3 × 3) matrix and explain what
          entry S[0][1] means conceptually.

- **(c)** Scale the similarity matrix by dividing by √d_k.
          What is the value of √d_k here?
          Show the full scaled matrix.

- **(d)** Apply Softmax **row-wise** to the scaled matrix.
          Use the formula:
```
softmax(x_i) = exp(x_i) / Σ exp(x_j)
```
          Show the full (3 × 3) attention weight matrix.
          Verify that each row sums to 1.0.

- **(e)** Compute the final output matrix `Z = Attention_weights × V`.
          What is the shape of Z?
          What does each row of Z represent?

<details>
<summary>💡 <b>Click to reveal answer</b></summary>

```
Given:
  seq_len = 3,  d_model = d_k = d_v = 4
  W_q = W_k = W_v = Identity matrix

(a) Q, K, V matrices:
    Since W_q = W_k = W_v = Identity:
      Q = X × I = X
      K = X × I = X
      V = X × I = X

    Q = K = V = [ [1.0, 0.0, 1.0, 0.0],
                  [0.0, 1.0, 0.0, 1.0],
                  [1.0, 1.0, 0.0, 0.0] ]

    Shape of each: (3 × 4)  ← (seq_len × d_k)

(b) Raw similarity matrix S = Q × Kᵀ:
    Kᵀ shape = (4 × 3), so S = (3×4) × (4×3) = (3×3)

    S[i][j] = dot product of row i of Q with row j of K

    S[0][0] = [1,0,1,0]·[1,0,1,0] = 1+0+1+0 = 2
    S[0][1] = [1,0,1,0]·[0,1,0,1] = 0+0+0+0 = 0
    S[0][2] = [1,0,1,0]·[1,1,0,0] = 1+0+0+0 = 1
    S[1][0] = [0,1,0,1]·[1,0,1,0] = 0+0+0+0 = 0
    S[1][1] = [0,1,0,1]·[0,1,0,1] = 0+1+0+1 = 2
    S[1][2] = [0,1,0,1]·[1,1,0,0] = 0+1+0+0 = 1
    S[2][0] = [1,1,0,0]·[1,0,1,0] = 1+0+0+0 = 1
    S[2][1] = [1,1,0,0]·[0,1,0,1] = 0+1+0+0 = 1
    S[2][2] = [1,1,0,0]·[1,1,0,0] = 1+1+0+0 = 2

    S = [ [2, 0, 1],
          [0, 2, 1],
          [1, 1, 2] ]

    S[0][1] = 0 means: token 1's query has ZERO similarity
    with token 2's key — token 2 is the least relevant token
    for token 1 and gets the lowest weight in its row
    (0.1863 in part d)

(c) Scaled similarity matrix S_scaled = S / √d_k:
    √d_k = √4 = 2.0

    S_scaled = S / 2.0

    S_scaled = [ [1.000, 0.000, 0.500],
                 [0.000, 1.000, 0.500],
                 [0.500, 0.500, 1.000] ]

(d) Row-wise Softmax to get attention weights A:

    Row 0: softmax([1.000, 0.000, 0.500])
      exp(1.000) = 2.7183
      exp(0.000) = 1.0000
      exp(0.500) = 1.6487
      sum        = 5.3670
      A[0] = [2.7183/5.3670, 1.0000/5.3670, 1.6487/5.3670]
           = [0.5065,        0.1863,         0.3072]
      sum check: 0.5065 + 0.1863 + 0.3072 = 1.0000

    Row 1: softmax([0.000, 1.000, 0.500])
      exp(0.000) = 1.0000
      exp(1.000) = 2.7183
      exp(0.500) = 1.6487
      sum        = 5.3670
      A[1] = [1.0000/5.3670, 2.7183/5.3670, 1.6487/5.3670]
           = [0.1863,        0.5065,         0.3072]
      sum check: 0.1863 + 0.5065 + 0.3072 = 1.0000

    Row 2: softmax([0.500, 0.500, 1.000])
      exp(0.500) = 1.6487
      exp(0.500) = 1.6487
      exp(1.000) = 2.7183
      sum        = 6.0157
      A[2] = [1.6487/6.0157, 1.6487/6.0157, 2.7183/6.0157]
           = [0.2741,        0.2741,         0.4519]

      NOTE: rows 0 and 1 have the same exp values but
      in different positions — this is correct since
      S_scaled[0] and S_scaled[1] contain the same three
      numbers, with the first two swapped

      sum check: 0.2741 + 0.2741 + 0.4519 = 1.0001 ≈ 1.0
                 (the extra 0.0001 comes only from rounding to 4 decimals)

    A = [ [0.5065, 0.1863, 0.3072],
          [0.1863, 0.5065, 0.3072],
          [0.2741, 0.2741, 0.4519] ]

(e) Output Z = A × V:
    Shape: (3×3) × (3×4) = (3×4)  ← same as input X

    Z[0] = 0.5065×[1,0,1,0] + 0.1863×[0,1,0,1] + 0.3072×[1,1,0,0]
         = [0.5065, 0,      0.5065, 0     ]
         + [0,      0.1863, 0,      0.1863]
         + [0.3072, 0.3072, 0,      0     ]
         = [0.8137, 0.4935, 0.5065, 0.1863]

    Z[1] = 0.1863×[1,0,1,0] + 0.5065×[0,1,0,1] + 0.3072×[1,1,0,0]
         = [0.1863, 0,      0.1863, 0     ]
         + [0,      0.5065, 0,      0.5065]
         + [0.3072, 0.3072, 0,      0     ]
         = [0.4935, 0.8137, 0.1863, 0.5065]

    Z[2] = 0.2741×[1,0,1,0] + 0.2741×[0,1,0,1] + 0.4519×[1,1,0,0]
         = [0.2741, 0,      0.2741, 0     ]
         + [0,      0.2741, 0,      0.2741]
         + [0.4519, 0.4519, 0,      0     ]
         = [0.7259, 0.7259, 0.2741, 0.2741]
      (adding the rounded weights gives 0.2741 + 0.4519 = 0.7260;
       at full precision the sum is 0.72593, i.e. 0.7259)

    Z = [ [0.8137, 0.4935, 0.5065, 0.1863],   ← contextual embedding token 1
          [0.4935, 0.8137, 0.1863, 0.5065],   ← contextual embedding token 2
          [0.7259, 0.7259, 0.2741, 0.2741] ]  ← contextual embedding token 3

    Shape of Z: (3 × 4)

    Each row of Z is the CONTEXTUAL EMBEDDING for that token —
    a weighted blend of all value vectors, where the weights
    were determined by how relevant each token's key was to
    that token's query. Unlike the original X where each token
    only contained its own information, each row of Z now
    contains information gathered from ALL tokens in the
    sequence, weighted by attention scores.
```
</details>

## **Q2. Attention Weights and Value Vectors**

You are given the following pre-computed **attention weight matrix**
for a 4-token sequence. Each row represents one token's attention
distribution over all tokens in the sequence:
```
Tokens:  ["The",  "cat",  "sat",  "down"]
Indices:    0        1       2       3

Attention weight matrix (already after Softmax):

          "The"  "cat"  "sat"  "down"
"The"  [ [0.70,  0.20,  0.05,  0.05],   ← row 0
"cat"    [0.10,  0.60,  0.20,  0.10],   ← row 1
"sat"    [0.05,  0.40,  0.40,  0.15],   ← row 2
"down" ] [0.05,  0.15,  0.50,  0.30] ]  ← row 3
```

The Value matrix is:
```
V = [ [1.0,  0.0],    ← value vector for "The"
      [0.0,  1.0],    ← value vector for "cat"
      [1.0,  1.0],    ← value vector for "sat"
      [0.0,  0.5] ]   ← value vector for "down"

Shape: (4 × 2),   d_v = 2
```

- **(a)** Verify that the attention weight matrix is valid.
          What two properties must every row satisfy?
          Check both properties for row 2.

- **(b)** Looking at the attention weight matrix, which token
          does "sat" (row 2) attend to most strongly?
          What does this tell us about the contextual
          relationship the model has learned?

- **(c)** Compute the contextual output vector for the token
          **"sat"** (row 2) using:
```
Z[2] = Σ attention_weight[2][j] × V[j]   for j = 0, 1, 2, 3
```
          Show all intermediate steps.

- **(d)** Compute the contextual output vector for the token
          **"cat"** (row 1).

- **(e)** Compare Z[1] ("cat") and Z[2] ("sat").
          Both tokens exist in the same sentence.
          Why are their output vectors different even though
          they use the same Value matrix?

- **(f)** A student claims: *"Since all rows use the same
          Value matrix V, all output vectors Z must be identical."*
          Is the student correct? Explain precisely why or why not.

<details>
<summary>💡 <b>Click to reveal answer</b></summary>

```
Given:
  4 tokens: ["The", "cat", "sat", "down"]
  Attention weights A: (4 × 4) — already post-Softmax
  Value matrix V: (4 × 2),  d_v = 2

(a) Two properties every valid attention weight row must satisfy:
    Property 1: All values must be NON-NEGATIVE (≥ 0)
                — they are probabilities, negative probabilities
                  are meaningless
    Property 2: All values in each row must SUM TO 1.0
                — they form a probability distribution over tokens

    Checking row 2 = [0.05, 0.40, 0.40, 0.15]:
      Non-negative check: 0.05, 0.40, 0.40 and 0.15 are all ≥ 0
      Sum check: 0.05 + 0.40 + 0.40 + 0.15 = 1.00
      Row 2 is a valid attention distribution

(b) "sat" (row 2) attention weights = [0.05, 0.40, 0.40, 0.15]
    Highest weights: "cat" (0.40) and "sat" itself (0.40) — tied

    This tells us the model has learned that "sat" is most
    strongly related to "cat" (its grammatical subject) and
    to itself. This makes linguistic sense: to understand what
    "sat" means in context, the model focuses on WHO is doing
    the sitting ("cat") and the action itself ("sat").
    "down" gets 0.15 (some relevance as a modifier) while
    "The" gets only 0.05 (the article is largely irrelevant
    to the verb "sat")

(c) Contextual output for "sat" — Z[2]:
    Z[2] = 0.05×V[0] + 0.40×V[1] + 0.40×V[2] + 0.15×V[3]

    0.05 × [1.0, 0.0] = [0.050, 0.000]
    0.40 × [0.0, 1.0] = [0.000, 0.400]
    0.40 × [1.0, 1.0] = [0.400, 0.400]
    0.15 × [0.0, 0.5] = [0.000, 0.075]
    ─────────────────────────────────
    Z[2]              = [0.450, 0.875]

    Verification:
    dim 0: 0.050 + 0.000 + 0.400 + 0.000 = 0.450
    dim 1: 0.000 + 0.400 + 0.400 + 0.075 = 0.875

(d) Contextual output for "cat" — Z[1]:
    Z[1] = 0.10×V[0] + 0.60×V[1] + 0.20×V[2] + 0.10×V[3]

    0.10 × [1.0, 0.0] = [0.100, 0.000]
    0.60 × [0.0, 1.0] = [0.000, 0.600]
    0.20 × [1.0, 1.0] = [0.200, 0.200]
    0.10 × [0.0, 0.5] = [0.000, 0.050]
    ─────────────────────────────────
    Z[1]              = [0.300, 0.850]

    Verification:
    dim 0: 0.100 + 0.000 + 0.200 + 0.000 = 0.300
    dim 1: 0.000 + 0.600 + 0.200 + 0.050 = 0.850

(e) Comparison of Z[1] and Z[2]:
    Z[1] ("cat")  = [0.300, 0.850]
    Z[2] ("sat")  = [0.450, 0.875]

    They are DIFFERENT because their attention weight ROWS
    are different:
      "cat" row  = [0.10, 0.60, 0.20, 0.10]
                   → heavily weights itself (0.60)
      "sat" row  = [0.05, 0.40, 0.40, 0.15]
                   → splits attention between "cat" and itself

    Even though both use the same V matrix, each token blends
    the value vectors with DIFFERENT weights — producing a
    unique contextual embedding that reflects each token's
    specific role and relationships in the sentence.
    This is the core power of self-attention: same V, different
    attention patterns → different contextual representations.

(f) The student is INCORRECT.
    The output vectors Z are computed as:
      Z[i] = Σ A[i][j] × V[j]

    While V is shared across all tokens, each token i has its
    OWN unique attention weight ROW A[i] — determined by how
    that token's Query matched every token's Key.
    Different attention rows → different weighted combinations
    of V → different output vectors.

    The outputs would only be identical if every row of A were
    identical — which would mean every token attends to all
    other tokens in exactly the same way, making attention
    completely useless.

    In practice, attention rows are almost always different because
    each token has a different query, capturing different
    contextual relationships — which is the entire purpose
    of self-attention.
```
</details>

## **Q3. Concepts: Q, K, V, Scaling and Masking**
- **(a)** In self-attention, every token produces three vectors Q, K, V.
        In ONE sentence explain what each of these three vectors represents
        conceptually (what role does each play in attention?)
- **(b)** Why do we SCALE the similarity scores by dividing by √d_k
        before applying Softmax? What problem does scaling solve?
- **(c)** The attention formula is:
```
Attention(Q, K, V) = Softmax( Q × Kᵀ / √d_k ) × V
```
A student says: *"The Softmax output tells us the actual meaning
of each token."* Is the student correct? What does the Softmax
output actually represent?

- **(d)** What is the key difference between:
    - **Masked Self-Attention** (used in the decoder)
    - **Regular Self-Attention** (used in the encoder)

<details>
<summary>💡 <b>Click to reveal answer</b></summary>

```
(a) Role of Q, K, V:

    Q (Query)  → "What am I looking for?"
                 Each token asks a question about what context it needs
                 Token "it" asks: "which noun does 'it' refer to?"

    K (Key)    → "What information do I have to offer?"
                 Each token advertises its content to other tokens
                 Token "cat" says: "I am a noun, an animal, a subject"

    V (Value)  → "What information do I actually pass along?"
                 The actual content a token shares when attended to
                 Token "cat" passes its rich semantic representation

    Attention = match Q against all Ks to find relevance,
                then use that relevance to weight the Vs

(b) Why scale by √d_k:
    Q × Kᵀ produces dot products that grow in MAGNITUDE
    as d_k increases (more dimensions = larger dot products)
    Large dot products → extreme Softmax inputs → Softmax output
    becomes very close to 0 or 1 (near one-hot distribution)
    → Gradients become near zero → vanishing gradient problem!

    Dividing by √d_k keeps dot product magnitudes in a reasonable
    range regardless of d_k → stable Softmax gradients → stable training
    Example: d_k=64 → divide by √64=8  │  d_k=512 → divide by √512≈22.6

(c) The student is INCORRECT.
    The Softmax output is an ATTENTION WEIGHT DISTRIBUTION —
    a set of probabilities (summing to 1.0) that tells the model:
    "pay THIS MUCH attention to each token when computing my output"

    It represents RELEVANCE SCORES between tokens, not meaning.
    The actual meaning/content comes from the V (Value) vectors —
    the Softmax weights determine HOW MUCH of each V to include
    in the final output via the weighted sum.

(d) Masked Self-Attention vs Regular Self-Attention:

    Regular Self-Attention (Encoder):
      Every token can attend to EVERY token in the sequence, including itself
      Token 3 can see tokens 1, 2, 3, 4, 5, 6... (past, itself AND future)
      Used in encoder — full context is available (e.g., translation input)

    Masked Self-Attention (Decoder):
      Each token can only attend to PREVIOUS tokens AND ITSELF (left context only)
      Token 3 can see tokens 1, 2 and 3, but NOT tokens 4, 5, 6...
      Future positions are masked with -∞ before Softmax → become 0
      Used in decoder — prevents the model from "cheating" by
      looking at future tokens it is supposed to GENERATE
```
</details>

## **Q4. Multi-Head Self-Attention**

The original transformer uses **h = 8 attention heads** with
**d_model = 512**.

**(a)** What is the dimension of Q, K, V per head (d_k and d_v)?
        Show the calculation.

**(b)** A student says:
        *"Multi-head attention is just running 8 separate attention
        operations and then averaging the results."*
        Identify TWO mistakes in this statement.

**(c)** What is the shape of the output from ONE attention head?
        (sequence length = 10 tokens)

**(d)** After running all 8 heads, the outputs are concatenated.
        What is the shape of the concatenated output?

**(e)** A final weight matrix W_O is applied after concatenation.
        What must the shape of W_O be, and what does it do?

**(f)** Why is multi-head attention better than single-head attention
        with d_k = 512? What can multiple heads capture that one
        head cannot?

<details>
<summary>💡 <b>Click to reveal answer</b></summary>

```
Given: h=8 heads, d_model=512, sequence_length=10

(a) d_k = d_v = d_model / h
              = 512 / 8
              = 64

    Each head works with 64-dimensional Q, K, V vectors
    (instead of the full 512 dimensions)

(b) Two mistakes in the student's statement:

    Mistake 1 — "averaging":
    The outputs are CONCATENATED, not averaged.
    concat([head1, head2, ..., head8]) → shape (seq × 512)
    Averaging would lose information from individual heads

    Mistake 2 — "just running 8 separate attention operations":
    It leaves out the two learned projections that make
    multi-head attention work:
      • BEFORE attention, each head projects the input into its
        own 64-dimensional subspace with its own W_Q_i, W_K_i,
        W_V_i (512 × 64 each), so each head can learn a
        different kind of relationship
      • AFTER concatenation, a learned output matrix W_O
        (512 × 512) mixes the information from all 8 heads

(c) Shape of ONE attention head output:
    Each head produces: (sequence_length × d_v)
                      = (10 × 64)

(d) Shape after concatenating all 8 heads:
    = (sequence_length × (h × d_v))
    = (10 × (8 × 64))
    = (10 × 512)

(e) Shape of W_O:
    Input to W_O  = (10 × 512)  →  W_O shape = (512 × 512)
    Output        = (10 × 512)  →  same as d_model

    What W_O does:
    Projects the concatenated multi-head output back into d_model
    space — mixes and combines the information from all 8 heads
    into one unified representation per token

(f) Why multi-head > single-head (d_k=512):

    Single head with d_k=512:
    One set of Q,K,V → ONE attention distribution per token
    Different relationships (subject-verb, coreference, nearby
    words) must all be blended into that single weighted average

    8 heads with d_k=64 each:
    Head 1 might learn: syntactic relationships (subject-verb)
    Head 2 might learn: coreference (pronoun → noun)
    Head 3 might learn: local context (adjacent words)
    Head 4 might learn: semantic similarity
    Head 5-8 might learn: other linguistic patterns

    Multiple heads allow the model to simultaneously attend to
    DIFFERENT aspects of the input from DIFFERENT representation
    subspaces — richer and more expressive than one big head
    Same total size: 8 heads × 64 = 512 dimensions, and the same number
    of parameters, because 8 × (512 × 64) = 512 × 512 (see Q7(a))
```
</details>


## **Q5. Masked Self-Attention**

You are in the decoder. The sequence so far is **3 tokens**.
The scaled similarity matrix before masking is:

```
         Token1  Token2  Token3
Token1 [  2.0     1.0     0.8  ]
Token2 [  1.5     2.5     1.2  ]
Token3 [  0.9     1.8     2.2  ]
```

**(a)** Apply the decoder mask. Write the masked matrix.
        *(Hint: future positions are replaced with -∞)*

**(b)** Apply Softmax row-wise to the masked matrix.
        Show the full calculation for Row 2 (Token 2) only.
        *(Use e^(-∞) = 0)*

**(c)** For Row 3 (Token 3), without full calculation, write what
        the Softmax output will look like qualitatively
        (which positions will be zero and which will be non-zero?)

**(d)** A student asks: *"Why do we use -∞ for masking instead
        of just putting 0?"* Give a precise explanation.

**(e)** Cross-attention in the decoder uses:
        - Q from the DECODER
        - K and V from the ENCODER

        Why? What would happen if K and V also came from the decoder?

<details>
<summary>💡 <b>Click to reveal answer</b></summary>

```
(a) Decoder mask — mask all positions ABOVE the diagonal with -∞:
    (Token i cannot attend to Token j if j > i)

             Token1  Token2   Token3
    Token1 [  2.0     -∞       -∞   ]   ← can only see itself
    Token2 [  1.5     2.5      -∞   ]   ← can see tokens 1,2
    Token3 [  0.9     1.8      2.2  ]   ← can see all 3 tokens

(b) Softmax for Row 2 (Token 2): [1.5, 2.5, -∞]

    Step 1 — exponentiate:
      e^1.5  = 4.4817
      e^2.5  = 12.1825
      e^(-∞) = 0.0000   ← masked position contributes nothing

    Step 2 — sum:
      sum = 4.4817 + 12.1825 + 0.0000 = 16.6642

    Step 3 — divide:
      A[2,1] = 4.4817  / 16.6642 = 0.2689
      A[2,2] = 12.1825 / 16.6642 = 0.7311
      A[2,3] = 0.0000  / 16.6642 = 0.0000  ← zero attention to future

    Row 2 of A = [0.2689, 0.7311, 0.0000]
    Check: 0.2689 + 0.7311 + 0.0000 = 1.0000

    Token 2 attends 73% to itself and 27% to Token 1
    and ZERO to Token 3 (future token — correctly masked!)

(c) Row 3 (Token 3) qualitative Softmax output:
    Input row = [0.9, 1.8, 2.2]  ← no masking needed for last token!
    ALL three positions are visible to Token 3 (no future tokens)

    Softmax output: [small, medium, largest]
    All three values will be NON-ZERO since no position is masked
    Token 3 attends most to itself (2.2 is highest)
    followed by Token 2 (1.8) then Token 1 (0.9)

(d) Why -∞ instead of 0:

    If we put 0 in masked positions:
      Softmax([2.0, 0, 0]):
        e^2 = 7.3891,  e^0 = 1,  e^0 = 1,  sum = 9.3891
        → [7.3891/9.3891, 1/9.3891, 1/9.3891] = [0.787, 0.107, 0.107]
      The masked positions get Softmax output 0.107 — NOT zero!
      The model still attends to future tokens with about 11% weight each!
      Masking with 0 FAILS to prevent future token attention

    If we put -∞ in masked positions:
      Softmax([2.0, -∞, -∞]) → [e^2/(e^2+0+0)] = [1.0, 0.0, 0.0]
      e^(-∞) = 0 exactly → masked positions get EXACTLY zero weight
      The model attends to future tokens with 0% weight

    -∞ is the mathematically correct way to force exactly zero
    attention weight after Softmax — it is the only value that
    guarantees complete masking regardless of the other scores
    (in code, a very large negative number such as -1e9 is often
    used instead; after Softmax its weight is 0 in practice)

(e) Why Cross-Attention uses Q from decoder, K and V from encoder:

    Q from decoder = "What does my current decoder state need?"
    K from encoder = "What information does each encoder token offer?"
    V from encoder = "What content does each encoder token contain?"

    The decoder QUERIES the encoder output for relevant context.
    For translation: decoder generating "chat" (French) queries
    the encoder's representation of "cat" (English) — it needs
    to know WHAT the encoder understood about the source sentence

    If K and V also came from the decoder:
    The decoder would only attend to its own partial output
    (the tokens generated so far) — it would NEVER access
    the encoder's understanding of the source sentence!
    The model would have no connection to the input — it would
    just be predicting the next token from previous predictions
    with no access to what it is supposed to be translating/answering
```
</details>


## **Q6. Attention Score Analysis — Putting It All Together**

You have a 3-token sequence: **"The cat sat"**
After computing scaled dot-product attention, the attention weight
matrix (after Softmax) for the ENCODER is:

```
            "The"   "cat"   "sat"
"The"    [  0.60    0.30    0.10  ]
"cat"    [  0.20    0.65    0.15  ]
"sat"    [  0.10    0.45    0.45  ]
```

**(a)** Verify that this is a valid attention weight matrix.
        What two properties must every row satisfy?

**(b)** Interpret the attention pattern for the word **"sat"**.
        Which token does "sat" attend to most? Why might this
        make linguistic sense?

**(c)** The Value vectors are:

```
V_The = [1.0, 0.0]
V_cat = [0.0, 1.0]
V_sat = [0.5, 0.5]
```

Compute the complete attention output matrix Z (3×2).
Show all three rows.

**(d)** Now suppose this were a DECODER with masked self-attention.
        Rewrite the attention weight matrix after masking is applied.
        *(Re-normalise each row after masking: masking before Softmax
        is the same as setting the masked weights to 0 and re-normalising.
        Row 2 needs the full calculation; state the results for rows 1 and 3)*

**(e)** Compare the encoder output Z (from part c) with what the
        decoder output would be for "sat" after masking.
        What information does "sat" lose in the decoder vs encoder?
        Why is this acceptable in a decoder?

<details>
<summary>💡 <b>Click to reveal answer</b></summary>

```
(a) Valid attention weight matrix properties:
    Property 1: Every value must be between 0 and 1 (non-negative)
    Property 2: Every ROW must sum to exactly 1.0 (probability dist)

    Check:
    Row 1: 0.60 + 0.30 + 0.10 = 1.00
    Row 2: 0.20 + 0.65 + 0.15 = 1.00
    Row 3: 0.10 + 0.45 + 0.45 = 1.00
    All values between 0 and 1  → Valid attention weight matrix

(b) "sat" attention pattern: [0.10, 0.45, 0.45]
    "sat" attends equally to "cat" (0.45) and itself (0.45)
    and very little to "The" (0.10)

    Linguistic interpretation:
    "sat" is a VERB — it needs to find its SUBJECT to understand
    the full meaning. "cat" is the subject of "sat" — the model
    has correctly learned that the verb should attend strongly
    to the noun that performs the action.
    Equal attention to itself (0.45) is also expected — a word
    always needs its own representation in the output.

(c) Z = A × V  (3×2 output matrix)

    Row 1 ("The"):  A = [0.60, 0.30, 0.10]
      dim0: 0.60×1.0 + 0.30×0.0 + 0.10×0.5 = 0.60 + 0.00 + 0.05 = 0.65
      dim1: 0.60×0.0 + 0.30×1.0 + 0.10×0.5 = 0.00 + 0.30 + 0.05 = 0.35
      Z_The = [0.65, 0.35]

    Row 2 ("cat"):  A = [0.20, 0.65, 0.15]
      dim0: 0.20×1.0 + 0.65×0.0 + 0.15×0.5 = 0.20 + 0.00 + 0.075 = 0.275
      dim1: 0.20×0.0 + 0.65×1.0 + 0.15×0.5 = 0.00 + 0.65 + 0.075 = 0.725
      Z_cat = [0.275, 0.725]

    Row 3 ("sat"):  A = [0.10, 0.45, 0.45]
      dim0: 0.10×1.0 + 0.45×0.0 + 0.45×0.5 = 0.10 + 0.00 + 0.225 = 0.325
      dim1: 0.10×0.0 + 0.45×1.0 + 0.45×0.5 = 0.00 + 0.45 + 0.225 = 0.675
      Z_sat = [0.325, 0.675]

    Complete output:
    Z = [[0.650, 0.350],    ← "The" output
         [0.275, 0.725],    ← "cat" output
         [0.325, 0.675]]    ← "sat" output

(d) Decoder masked attention:
    The given matrix contains attention WEIGHTS (after Softmax),
    not scores, so we must NOT apply Softmax to them again.

    Key fact: masking a position before Softmax is the same as
    setting its weight to 0 and RE-NORMALIZING the remaining
    weights so that the row sums to 1 again:
        softmax(masked scores)_j = e^(s_j) / Σ_(k≤i) e^(s_k)
                                 = A[i][j] / Σ_(k≤i) A[i][k]

    Row 1 ("The") — keep only "The": [0.60, ✗, ✗]
      A_masked[1] = [0.60/0.60, 0, 0] = [1.0000, 0.0000, 0.0000]
      ("The" can only attend to itself)

    Row 2 ("cat") — keep "The" and "cat": [0.20, 0.65, ✗]
      sum of visible weights = 0.20 + 0.65 = 0.85
      A_masked[2] = [0.20/0.85, 0.65/0.85, 0]
                  = [0.2353, 0.7647, 0.0000]
      Check: 0.2353 + 0.7647 = 1.0000
      ("cat" attends to "The" and itself only)

    Row 3 ("sat") — no masking needed (last token sees all):
      A_masked[3] = [0.10, 0.45, 0.45]  ← unchanged

(e) Comparison — "sat" in encoder vs decoder:

    Encoder "sat" output: Z_sat = [0.325, 0.675]
    Uses full attention: [0.10, 0.45, 0.45]
    → "sat" has access to "The" (0.10), "cat" (0.45), itself (0.45)
    → Full bidirectional context → richest representation

    Decoder "sat" output: Z_sat = same [0.325, 0.675] here
    (since "sat" is last token — no future tokens to mask)
    BUT for earlier tokens like "cat":

    Encoder "cat": [0.275, 0.725] — attended to all 3 tokens
    Decoder "cat": 0.2353×[1.0, 0.0] + 0.7647×[0.0, 1.0]
                 = [0.2353, 0.7647] — attended to only "The" and itself
    → Cannot see "sat" which comes AFTER it in sequence
    → Less rich representation — missing future context

    Why this is acceptable in the decoder:
    The decoder generates text LEFT TO RIGHT at inference time —
    when generating token 2 ("cat"), token 3 ("sat") does NOT
    EXIST YET. The model cannot cheat by looking at words it
    has not generated yet. The masking enforces the autoregressive
    property — generate one token at a time using only the past.
    During training, masking simulates this inference condition
    so the model learns to predict without future information.
```
</details>


## **Q7. Multi-Head Attention**

The original transformer uses h=8 heads and d_model=512.
Consider the following alternative designs:

```
Design A (original): h=8,  d_model=512, d_k=64
Design B           : h=1,  d_model=512, d_k=512
Design C           : h=16, d_model=512, d_k=32
Design D           : h=8,  d_model=512, d_k=128  ← student's proposal
```

**(a)** Calculate the total number of parameters in the Q projection
        matrices (W_Q for all heads combined) for each design.
        *(Each head has its own W_Q of shape d_model × d_k)*

**(b)** Design D has d_k=128 with h=8 heads, so h × d_k ≠ d_model.
        Does it still work? What does it cost?
        Is h × d_k = d_model a strict rule?

**(c)** Calculate the total FLOPs (floating point operations) for
        computing Q×Kᵀ for ONE head in Design A vs Design B,
        for a sequence of **100 tokens**.
        *(Matrix multiply of shape (n×d_k) × (d_k×n) costs 2×n×n×d_k FLOPs)*

**(d)** A researcher proposes Design C (16 heads, d_k=32).
        - How does it compare to Design A in total parameters?
        - What is the risk of using very small d_k=32 per head?
        - When might Design C be preferred over Design A?

**(e)** The total self-attention computation scales as O(n²×d) where
        n is sequence length and d is d_model.
        For a sequence of n=1000 tokens vs n=100 tokens (d_model=512):
        - How many times more computation does n=1000 require?
        - Why is this quadratic scaling a major problem for LLMs?
        - Name ONE modern technique that addresses this problem.

<details>
<summary>💡 <b>Click to reveal answer</b></summary>

```
(a) Parameters in W_Q matrices (all heads combined):

    Each head has W_Q shape = (d_model × d_k)

    Design A (h=8,  d_k=64):
      Per head    = 512 × 64  = 32,768
      All 8 heads = 8 × 32,768 = 262,144 params

    Design B (h=1,  d_k=512):
      Per head    = 512 × 512 = 262,144
      1 head      = 262,144 params

    Design C (h=16, d_k=32):
      Per head    = 512 × 32  = 16,384
      All 16 heads = 16 × 16,384 = 262,144 params

    Design D (h=8,  d_k=128):
      Per head    = 512 × 128 = 65,536
      All 8 heads = 8 × 65,536 = 524,288 params

    Interesting: A, B, C all have IDENTICAL parameter counts (262,144)!
    Only Design D has more parameters.

(b) Design D (h=8, d_k=d_v=128):

    After concatenating all heads:
    Output shape = h × d_v = 8 × 128 = 1024, which is NOT d_model (512)

    Does it work?  YES. W_O has shape (h·d_v × d_model), so here
    W_O = (1024 × 512) projects the concatenated output back to 512.
    Nothing in the math forces h × d_k = d_model.

    What does it cost?
    - W_Q, W_K, W_V each double (524,288 params instead of 262,144)
    - W_O doubles (1024 × 512 instead of 512 × 512)
    - the attention computation doubles (8 heads with d_k = 128)

    So h × d_k = d_model is a DESIGN CHOICE (a convention), not a rule.
    The paper chose it so that multi-head attention costs about the
    same as single-head attention with full dimensionality:
    Design A: 8 × 64  = 512 = d_model     Design B: 1 × 512 = 512 = d_model
    Design C: 16 × 32 = 512 = d_model     Design D: 8 × 128 = 1024 (works, 2× the cost)

    Real models sometimes break the convention, e.g., Gemma-7B has
    d_model = 3072 but 16 heads × 256 = 4096.

(c) FLOPs for Q×Kᵀ (sequence n=100):

    Formula: 2 × n × n × d_k FLOPs

    Design A (one head, d_k=64):
      = 2 × 100 × 100 × 64
      = 2 × 10,000 × 64
      = 1,280,000 FLOPs = 1.28M FLOPs per head
      For all 8 heads = 8 × 1.28M = 10.24M FLOPs

    Design B (one head, d_k=512):
      = 2 × 100 × 100 × 512
      = 2 × 10,000 × 512
      = 10,240,000 FLOPs = 10.24M FLOPs for the single head

    Design A total (8 heads): 10.24M FLOPs
    Design B total (1 head):  10.24M FLOPs
    → IDENTICAL total computation!
    Multi-head attention has same compute as single-head
    but richer representations — no free lunch but no extra cost!

(d) Design C analysis (h=16, d_k=32):

    Total parameters: same as Design A (262,144) ← calculated above
    Compute: same as Design A (16 × 2×100×100×32 = 10.24M)

    Risk of very small d_k=32:
    Each head only has 32 dimensions to work with
    → Very limited capacity per head to represent relationships
    → Each head may be too "narrow" to capture complex patterns
    → Diminishing returns — 16 very weak heads may underperform
       8 moderate heads

    When to prefer Design C:
    • When you want more diverse specialization across more heads
    • When tasks have many distinct linguistic relationship types
    • When d_k=64 is empirically shown to be overkill for the task
    (Parameters and compute are the same as Design A, so the
     choice is about quality, not about memory)

(e) Quadratic scaling O(n²×d):

    n=100 tokens:  compute ∝ 100²  × 512 = 5,120,000
    n=1000 tokens: compute ∝ 1000² × 512 = 512,000,000

    Ratio = 1000² / 100² = 1,000,000 / 10,000 = 100×

    n=1000 requires 100× more computation than n=100!
    (Doubling sequence length → 4× more computation)

    Why this is a major problem for LLMs:
    Modern LLMs need context windows of 8K, 32K, 128K tokens
    At 128K tokens: (128,000)² = 16.4 BILLION query-key pairs,
    i.e., about 8.4 TRILLION multiply-adds for Q×Kᵀ (with d = 512)
    just for ONE attention layer!
    Memory also scales as O(n²) if the attention matrix is stored:
    128K × 128K attention matrix = 16.4 billion numbers
                                 ≈ 33 GB in 16-bit (≈ 66 GB in 32-bit)
                                 for ONE head of ONE layer!

    Modern techniques that address quadratic scaling:
    Any ONE of the following is acceptable:
    • Flash Attention    — computes EXACT attention in tiles without storing
                           the full n×n matrix: memory O(n) instead of O(n²)
                           (compute is still O(n²), but much faster in practice)
    • Sliding Window     — each token attends to local window only
    • Sparse Attention   — attend to subset of positions
    • Linear Attention   — approximates attention in O(n) time
    • Ring Attention     — splits the sequence across devices
    ⚠️ GQA / MQA are NOT an answer here: they shrink the KV cache
       (memory), but the attention compute is still O(n²) (see §12)
```
</details>


## **Q8. Cross-Attention**

You are building a translation model (English → French).
The encoder processes the English sentence:
**"The cat sat"** (3 tokens)

The encoder output (after all encoder layers) is:

```
Encoder output H:
H_The = [1.0, 0.0]
H_cat = [0.0, 1.0]
H_sat = [0.5, 0.5]
```

The decoder has generated one token so far: **"Le"**
The decoder's current hidden state is:

```
Decoder state S_Le = [0.3, 0.7]
```

In cross-attention:
- **Q comes from the decoder** (projected from S_Le)
- **K and V come from the encoder** (projected from H)

Assume all projection matrices are identity (W_Q = W_K = W_V = I)
and d_k = 2.

**(a)** Write down the Q vector (from decoder) and the
        K, V matrix (from encoder)

**(b)** Compute the raw similarity scores between the decoder
        query Q and all 3 encoder keys K.
        *(3 dot products)*

**(c)** Scale the scores by √d_k and apply Softmax.
        Show full calculation.

**(d)** Compute the cross-attention output for "Le".
        What does this output vector represent?

**(e)** Interpret the result:
        - Which English word does "Le" attend to most?
        - Does this make linguistic sense for English → French translation?
        - What would it mean if "Le" attended equally to all 3 words?

**(f)** Now suppose the decoder has generated two tokens: "Le chat"
        In the DECODER'S masked self-attention (before cross-attention):
        - Draw the attention mask for ["Le", "chat"]
        - Can "Le" attend to "chat"? Can "chat" attend to "Le"?
        - After masked self-attention, does cross-attention change?
          (i.e., do K and V change when decoder generates more tokens?)

<details>
<summary>💡 <b>Click to reveal answer</b></summary>

```
Given:
  Encoder output H = [[1.0, 0.0],   ← H_The
                       [0.0, 1.0],   ← H_cat
                       [0.5, 0.5]]   ← H_sat
  Decoder state S_Le = [0.3, 0.7]
  W_Q = W_K = W_V = Identity, d_k = 2

(a) Q vector (from decoder):
    Q_Le = S_Le × W_Q = [0.3, 0.7] × I = [0.3, 0.7]

    K matrix (from encoder — 3 key vectors):
    K_The = H_The × W_K = [1.0, 0.0]
    K_cat = H_cat × W_K = [0.0, 1.0]
    K_sat = H_sat × W_K = [0.5, 0.5]

    V matrix (from encoder — same as K here since W_V = I):
    V_The = [1.0, 0.0]
    V_cat = [0.0, 1.0]
    V_sat = [0.5, 0.5]

(b) Raw similarity scores (Q · K for each encoder position):

    score("The") = Q_Le · K_The = [0.3,0.7]·[1.0,0.0] = 0.3×1 + 0.7×0 = 0.30
    score("cat") = Q_Le · K_cat = [0.3,0.7]·[0.0,1.0] = 0.3×0 + 0.7×1 = 0.70
    score("sat") = Q_Le · K_sat = [0.3,0.7]·[0.5,0.5] = 0.3×0.5 + 0.7×0.5 = 0.50

    Raw scores = [0.30, 0.70, 0.50]

(c) Scale by √d_k = √2 ≈ 1.414214:

    Scaled scores = [0.30/√2, 0.70/√2, 0.50/√2]
                  = [0.212132, 0.494975, 0.353553]
                  ≈ [0.2121, 0.4950, 0.3536]

    Softmax (results rounded to 4 decimals):
      e^0.212132 = 1.2363
      e^0.494975 = 1.6405
      e^0.353553 = 1.4241
      sum        = 4.3009

      A_The = 1.2363 / 4.3009 = 0.2875
      A_cat = 1.6405 / 4.3009 = 0.3814
      A_sat = 1.4241 / 4.3009 = 0.3311

      Attention weights = [0.2875, 0.3814, 0.3311]
      Check: 0.2875 + 0.3814 + 0.3311 = 1.0000

(d) Cross-attention output for "Le":

    Z_Le = A · V   (weights kept to 6 decimals here, so that the
                    rounding does not show up in the result)
         = 0.287455×[1.0,0.0] + 0.381423×[0.0,1.0] + 0.331122×[0.5,0.5]

    dim0: 0.287455×1.0 + 0.381423×0.0 + 0.331122×0.5 = 0.287455 + 0 + 0.165561 = 0.453016 ≈ 0.4530
    dim1: 0.287455×0.0 + 0.381423×1.0 + 0.331122×0.5 = 0 + 0.381423 + 0.165561 = 0.546984 ≈ 0.5470

    Z_Le = [0.4530, 0.5470]

    What it represents:
    A context-aware representation for generating the French token "Le"
    that is a WEIGHTED MIX of all English encoder states — the decoder
    has looked at the entire English sentence and extracted the most
    relevant information to decide what French token to generate next

(e) Interpretation:

    Attention weights:
      "The" → 0.2875  (28.75%)
      "cat" → 0.3814  (38.14%) ← highest attention
      "sat" → 0.3311  (33.11%)

    "Le" attends MOST to "cat" (38.14%), but the three weights
    (0.29, 0.38, 0.33) are close to uniform. This is expected in a
    toy example: with identity projection matrices and tiny 2-D
    vectors there are no learned W_Q, W_K to sharpen the attention.

    Linguistic view:
      "Le" in French means "The" (masculine article)
      → A trained model would attend strongly to "The", which "Le"
        translates, and also to "cat", because "chat" is masculine
        and decides that the article is "Le" and not "La".
      → Trained W_Q and W_K make such patterns much sharper.

    If "Le" attended equally (0.333 each):
    The output would be a plain average of the three encoder vectors:
    the source tokens get blurred together, and the decoder gets no
    hint about WHICH source word matters for this step. Trained
    models usually show sharper patterns for this reason.

(f) Masked self-attention for decoder ["Le", "chat"]:

    Attention mask:
              "Le"   "chat"
    "Le"   [  ✅      ❌  ]   ← "Le" cannot see "chat" (future)
    "chat" [  ✅      ✅  ]   ← "chat" can see "Le" and itself

    As a matrix with -∞ masking:
              "Le"   "chat"
    "Le"   [ score    -∞  ]
    "chat" [ score   score]

    Can "Le" attend to "chat"?   ❌ NO — "chat" is a future token
    Can "chat" attend to "Le"?   ✅ YES — "Le" is a past token

    Does cross-attention K and V change?
    ❌ NO — K and V in cross-attention ALWAYS come from the
    ENCODER output H which is computed ONCE and stays FIXED
    for the entire decoding process regardless of how many
    decoder tokens have been generated.

    Only Q changes (it comes from the current decoder state)
    → As the decoder generates more tokens, Q changes
    → But K and V from the encoder remain the same
    → This is the key mechanism: the encoder "stores" the full
       source sentence and the decoder repeatedly queries it
    → So the cross-attention K and V can be computed ONCE and
       cached for the whole generation (see §12)
```
</details>

<br><br><br><br><br><br><br><br><br><br><br><br><br>
# <span style='background :lightgreen' >12. KV Cache</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The <b>KV cache</b> stores the Key and Value vectors of the tokens that have already been processed, so that during text generation (inference) each new token only needs to compute its own Q, K and V.</div></h3>

<div style="background-color:#dae8fc; border-left:8px solid #6c8ebf; padding:10px 14px; margin:24px 0px 8px 0px; font-size:1.3em;"><b>WHY</b> do we need a KV cache?</div>

## a. The Problem: Generation Repeats Work
- A decoder-only model generates text one token at a time. Every predicted token is appended to the input, and the model runs again to predict the next one.
- Our example in this section is the famous quote *"Ask not what your country can do for you – ask what you can do for your country."* The prompt is the 8 tokens **"Ask not what your country can do for"**. In step 1 the model reads them and predicts **"you"**. In step 2 it reads 9 tokens and predicts **"ask"**, and so on, until step 9 predicts **"country"**. (To keep the counting simple, we leave out "–" and ".", which are tokens too.)
- Without a cache, every step runs **all** the tokens through every layer again (Q, K, V, attention and FFN), although the results for the earlier tokens are **exactly the same** as in the step before. Only the newest token is new, and only its output is used to predict the next token.

<div style="text-align:center;">
    <img src="../images/kv-cache-08.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

- The cost grows with every step: the 9 steps process 8 + 9 + … + 16 = **108** tokens in every layer, although only **16** of them are new. For a 1,000-token prompt and a 1,000-token answer it is about **1.5 million** against **2,000**, i.e., 750× more work.

<br><br><br><br>

<div style="background-color:#dae8fc; border-left:8px solid #6c8ebf; padding:10px 14px; margin:24px 0px 8px 0px; font-size:1.3em;"><b>WHAT</b> is the KV cache?</div>

## b. The Idea: Remember the Keys and Values
- To predict the next token, the model only needs the output of the **newest** token. In each head, that output is the last row of the attention output of §5 (Step 5), and it needs:
    - the **query of the new token** (one row, 1 × 64 per head)
    - the **keys of all tokens**, to compute the scores
    - the **values of all tokens**, to compute the weighted sum
- The keys and values of the earlier tokens were already computed in the earlier steps. The **KV cache** stores them, in every layer and every head. Each step then computes q, k and v only for the new token, appends its k and v to the cache, and reads all the other keys and values from the cache.

<div style="text-align:center;">
    <img src="../images/kv-cache-09.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

<br>

<div style="text-align:center;">
    <img src="../images/kv-cache-01.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

<br><br><br><br>

## c. Why Caching Is Allowed
- **Why do the old K and V never change?** Because of the causal mask (§5), a token never attends to the tokens on its right. So when "you" is appended, the vectors of "Ask … for" stay the same in every layer, and so do their K and V. They can be computed once and kept in the cache.
- **Why don't we cache Q?** The query of an old token is only used to compute *that token's own* output, which was already computed in an earlier step and is never needed again. At each step only the query of the newest token is needed.

<div style="text-align:center;">
    <img src="../images/kv-cache-03.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

<br><br><br><br>

## d. What Exactly Is Stored
- For **every layer** and **every K, V head**, the cache keeps one K matrix and one V matrix, with one row per token. Each new token adds one row to all of them.
- So the cache of one sequence has the shape **layers × 2 (K and V) × KV heads × tokens × d_head**, and a batch has one such cache for every sequence.

<div style="text-align:center;">
    <img src="../images/kv-cache-05.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

**When and where is the KV cache used?**
- **Only during inference, never during pre-training or fine-tuning.** In training, the whole text is known in advance, so all positions are processed together in one parallel pass with the causal mask. There is no token-by-token loop, so nothing is computed twice and nothing needs to be cached. (Training methods in which the model generates text, such as RLHF, use the cache only while generating.)
- **Only in the decoder, never in the encoder.** The cache exists because the decoder runs once for every generated token, and its causal mask keeps the old K and V fixed. The encoder reads the whole input once, in one parallel pass, and is not run again while the decoder generates, so there is nothing to cache. Its attention is also bidirectional (as in BERT): every token attends to every other token, so adding a token would change all the earlier vectors.
- **Encoder-decoder models** (the original Transformer, BART, T5): the decoder's masked self-attention uses a KV cache that grows by one token per step, exactly as described above. The K and V of its **cross-attention** come from the encoder output, which does not change while the decoder is generating, so they are computed **once** per input and reused at every step (see Q8(f)).

**Three more facts about the cache**
- It belongs to **one** request: it lives in GPU memory while the text is being generated and is thrown away afterwards. It is not part of the model weights.
- It is **exact**: the model gives the same output as when everything is recomputed (we check this in §f).
- **KV cache vs prompt caching:** API providers (OpenAI, Anthropic, Google) offer *prompt caching*: they keep the K and V of a long prefix, e.g., a system prompt, **between** requests. A later request that starts with the same prefix skips its prefill, which is why cached input tokens are cheaper and faster.

<br><br><br><br>

<div style="background-color:#dae8fc; border-left:8px solid #6c8ebf; padding:10px 14px; margin:24px 0px 8px 0px; font-size:1.3em;"><b>HOW</b> does it work?</div>

## e. How It Works: Prefill and Decode
- **Prefill** (step 1): the whole prompt "Ask not what your country can do for" is processed in one parallel pass, exactly as in §4 and §5. The model predicts the first output token, "you", and **fills the cache** with the K and V of all 8 prompt tokens, in every layer and every head. Its time is the **time to first token (TTFT)**.
- **Decode** (steps 2 to 9): from then on, the model processes **one new token per step**, the token it has just predicted. It appends the k and v of this token to the cache and predicts the next token. The time of one such step is the **time per output token**.

<div style="text-align:center;">
    <img src="../images/kv-cache-06.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

- Step by step for our quote:

| Step | Phase | Token(s) fed in | Predicted token | Tokens processed without cache | Tokens processed with cache | Tokens in the cache after the step |
|---|---|---|---|---|---|---|
| 1 | prefill | Ask not what your country can do for | you | 8 | 8 | 8 |
| 2 | decode | you | ask | 9 | 1 | 9 |
| 3 | decode | ask | what | 10 | 1 | 10 |
| 4 | decode | what | you | 11 | 1 | 11 |
| 5 | decode | you | can | 12 | 1 | 12 |
| 6 | decode | can | do | 13 | 1 | 13 |
| 7 | decode | do | for | 14 | 1 | 14 |
| 8 | decode | for | your | 15 | 1 | 15 |
| 9 | decode | your | country | 16 | 1 | 16 |
| **Total** | | | | **108** | **16** | |

- In every decode step, the new token goes up through all the layers, and in each layer its query reads that layer's cache:

<div style="text-align:center;">
    <img src="../images/kv-cache-04.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

- Inside each layer and each head, a decode step does the following (x_new is the vector of the new token at that layer, and t is the number of tokens in the cache before the step):
```
q_new   = x_new × W_q                            (1 × 64)
k_new   = x_new × W_k ,   v_new = x_new × W_v    (1 × 64 each)
K_cache = append k_new to K_cache                (t × 64)  →  ((t+1) × 64)
V_cache = append v_new to V_cache                (t × 64)  →  ((t+1) × 64)
scores  = q_new × K_cacheᵀ / √d_k                (1 × (t+1))
output  = softmax(scores) × V_cache              (1 × 64)
```

<div style="text-align:center;">
    <img src="../images/kv-cache-02.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

- No mask is needed in a decode step: the cache only contains the past tokens and the new token itself.
- **Positions:** the new token still gets its own position. In step 2, "you" is token number 9, so it gets the positional encoding of position 9, not of position 1. The cached keys keep the positions they were computed with.
- Prefill is **compute-bound** (large matrix multiplications over many tokens), while decode is **memory-bandwidth-bound**: each step does little computation but must read all the model weights and the whole KV cache from GPU memory. This is why the size of the cache matters so much.
- For example, Llama-3.1-8B has about 16 GB of weights in 16-bit numbers. An H100 GPU reads about 3.35 TB per second from its memory, so just reading the weights once takes about 5 ms: one sequence can never get more than about 200 tokens per second, however fast the GPU computes.
- In Hugging Face Transformers the cache is switched on by default (`use_cache=True`) and is returned by the model as `past_key_values`.

## f. KV-cache vs. no-KV-cache text generation with GPT-2

In [11]:
import threading
from time import perf_counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer

# ------------------------------------------------------------------
# 1. Load model and tokenizer
#    Note: plain AutoModel loads GPT2Model, which has no language-model
#    head and cannot call .generate(). AutoModelForCausalLM is the
#    AutoModel class meant for text generation.
# ------------------------------------------------------------------
MODEL_NAME = "gpt2"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

# GPT-2 has no pad token; reuse EOS to avoid warnings during generate()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

prompt = "Once upon a time," #"The next day is bright"
inputs = tokenizer(prompt, return_tensors="pt").to(device)  # input_ids + attention_mask

MAX_NEW_TOKENS = 100   #500


# ------------------------------------------------------------------
# 2. Streaming generation in a background thread
# ------------------------------------------------------------------
def stream_output(use_cache: bool) -> float:
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,          # print only the newly generated text
        skip_special_tokens=True,
        timeout=120.0,             # don't hang forever if generation fails
    )

    generation_kwargs = dict(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=MAX_NEW_TOKENS,
        min_new_tokens=MAX_NEW_TOKENS,   # force the same length in both runs
        do_sample=False,                 # greedy -> identical text, fair timing
        no_repeat_ngram_size=3,          # stop GPT-2 from looping on one sentence
        use_cache=use_cache,
        pad_token_id=tokenizer.eos_token_id,
        streamer=streamer,
    )

    thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)

    print(prompt, end="", flush=True)
    start_time = perf_counter()
    thread.start()

    for text_chunk in streamer:
        print(text_chunk, end="", flush=True)

    thread.join()
    elapsed = perf_counter() - start_time

    print(f"\n\nUse cache = {use_cache}, Time taken: {elapsed:.3f} seconds "
          f"({MAX_NEW_TOKENS / elapsed:.1f} tokens/sec)\n")
    return elapsed


# ------------------------------------------------------------------
# 3. Warm-up (first call pays one-time setup costs; don't count it)
# ------------------------------------------------------------------
with torch.no_grad():
    model.generate(**inputs, max_new_tokens=5, pad_token_id=tokenizer.eos_token_id)


# ------------------------------------------------------------------
# 4. Run both scenarios
# ------------------------------------------------------------------
if __name__ == "__main__":
    print(f"Device: {device}\n")

    print("=== With KV caching ===")
    t_cache = stream_output(use_cache=True)

    print("=== Without KV caching ===")
    t_no_cache = stream_output(use_cache=False)

    print("=== Summary ===")
    print(f"With KV cache    : {t_cache:.3f} s")
    print(f"Without KV cache : {t_no_cache:.3f} s")
    print(f"Speed-up         : {t_no_cache / t_cache:.2f}x faster with KV cache")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Device: cpu

=== With KV caching ===
Once upon a time, the world was a place of great beauty and great danger. The world was filled with the most beautiful creatures, and the most dangerous creatures. The people of the world were the most fearful, and they were the ones who were most afraid. The most fearful people were the people of darkness, and darkness was the most powerful. The darkness was a great power, and it was the power of darkness that was the greatest. The dark power was the darkness that had the greatest power. The power of the darkness was that of darkness. The light power was that which was the light that was most powerful, and that was that power of light that had most powerful power.

The darkness was not the power that was strongest, but the power which was most dangerous. The evil power was not that which could be overcome, but that which would be overcome. The good power was evil, and evil was the evil power. Evil was the good power, but evil was evil. The bad powe

<br><br><br><br>

<div style="background-color:#dae8fc; border-left:8px solid #6c8ebf; padding:10px 14px; margin:24px 0px 8px 0px; font-size:1.3em;"><b>THE PRICE</b> of the KV cache, and how to make it smaller</div>

## g. The Price Tag: Memory
- Speed costs memory. For every token, every layer stores one key vector and one value vector for each KV head (figure in §d), so:

$$\text{KV cache size} = 2 \times n_{layers} \times n_{kv\_heads} \times d_{head} \times \text{bytes per number} \times \text{sequence length} \times \text{batch size}$$

- The factor 2 is for K and V.
- **Example: Llama-3.1-8B** (32 layers, 8 KV heads, $d_{head}$ = 128, 16-bit numbers = 2 bytes):
```
per token      = 2 × 32 × 8 × 128 × 2 bytes  = 131,072 bytes = 128 KiB
128K tokens    = 128 KiB × 131,072 tokens    = 16 GiB   (one sequence, batch size 1)
model weights  ≈ 8.03 billion params × 2 bytes ≈ 16 GB
```
- At its full 128K context, the cache of a **single** conversation is about as large as the model itself, and serving 10 such conversations at once needs about 160 GiB just for the cache. For long contexts **the KV cache, not the model, becomes the bottleneck**.
- If Llama-3.1-8B used standard multi-head attention (32 KV heads, one per query head), the cache would be 4× larger: 512 KiB per token and 64 GiB at 128K tokens. The next section shows how modern models avoid this.
- **How many conversations fit on one GPU?** Turn the formula around. First compute the cache of **one** sequence, then:
    - largest batch = ⌊ memory free for the cache ÷ cache per sequence ⌋, where ⌊ ⌋ means round **down** (a sequence either fits completely or not at all)
    - Example: with 40 GiB free and 8,192-token conversations, Llama-3.1-8B needs 128 KiB × 8,192 = 1 GiB per conversation, so it can serve **40** conversations at the same time. With MHA (512 KiB × 8,192 = 4 GiB each) it could serve only **10**.
    - So the KV cache, and not the compute, often decides how many users one GPU can serve.

<br><br><br><br>

## h. Making the Cache Smaller
The formula of §h shows what can be made smaller: the number of K, V heads, the bytes per number, or the number of tokens stored. Four ideas are used today:
- **(i) Store fewer K, V heads: share them or compress them**
    - **MHA** (multi-head attention, the original Transformer): every query head has its own K, V head.
    - **GQA** (grouped-query attention): the query heads are split into groups, and each group shares one K, V head. **GQA-g** has g groups, so n_kv_heads = g. Llama-3.1-8B is GQA-8: its 32 query heads share 8 K, V heads, so its cache is 4× smaller than with MHA. Most models today use GQA.
    - **MQA** (multi-query attention): all query heads share one K, V head. The smallest cache of the three, but some loss in quality.
    - **MLA** (multi-head latent attention, DeepSeek): stores one small compressed vector per token, and rebuilds the K and V of all heads from it when they are needed.

<div style="text-align:center;">
    <img src="../images/kv-cache-07.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

- **(ii) Store smaller numbers (quantization):** keep K and V in 8 or 4 bits instead of 16 bits, like saving a photo as a JPEG instead of a RAW file.
- **(iii) Store fewer tokens (sliding window, eviction):** keep only the most recent tokens, plus a few important ones. This saves memory, but the model can no longer see the tokens that were thrown away.
- **(iv) Waste less memory (PagedAttention, used by vLLM):** store the cache in small blocks, like the pages of an operating system, so that no memory is reserved and left unused, and requests that start with the same prompt can share blocks.
- Ideas (i), (ii) and (iv) shrink the **memory** of the cache, but not the attention computation: every query head still compares its query with every cached key. Only (iii) also saves computation, because fewer keys are left to compare with.

## **Q9. KV Cache**

A small decoder-only model has **4 layers** with **8 query heads** each, and head size **d_head = 64**. K and V are stored as 16-bit numbers (**2 bytes** each). Use the formula of §12h:
```
KV cache size = 2 × n_layers × n_kv_heads × d_head × bytes per number × sequence length × batch size
1 KiB = 1,024 bytes        1 MiB = 1,024 KiB
```

**(a)** The prompt has **5 tokens**. Step 1 (prefill) processes the prompt and predicts token 6. Steps 2, 3 and 4 (decode) each process the newest token. For **one layer and one head**, how many rows of K are computed in each step **without** and **with** a KV cache? What are the two totals? (compare with the output of the code in §12f)

**(b)** In step 2 with a KV cache, put each of these into one of three groups: *computed in this step*, *read from the cache*, *not needed at all*.
- the q of token 6
- the k and v of token 6
- the k and v of tokens 1–5
- the q of tokens 1–5

**(c)** After step 4, how many rows does the K cache of one head in one layer have? If every query head has its own K, V head (MHA), how many K matrices and how many V matrices does the whole cache hold? (see the figure in §12d)

**(d)** With **MHA** (8 K, V heads per layer), how many bytes of cache does one token need? How large is the cache of one sequence of **1,024 tokens**?

**(e)** Repeat (d) for **GQA-2** (2 K, V heads, each shared by 4 query heads) and for **MQA** (1 K, V head). (see the figure in §12i)

**(f)** **64 MiB** of GPU memory is free for the cache, and every sequence has 1,024 tokens. How many sequences fit at the same time with MHA, GQA-2 and MQA?

**(g)** True or false?
1. Caching Q as well would make generation faster.
2. With a KV cache the model may generate a slightly different text than without it.
3. The K and V of earlier tokens never change because of the causal mask.
4. The KV cache is also used during pre-training.
5. GQA makes the cache smaller, but every query head still attends to all cached keys.
6. In an encoder-decoder model such as T5, the encoder also keeps a KV cache.

<details>
<summary>💡 <b>Click to reveal answer</b></summary>

```
(a) Without a cache, every step computes K for ALL tokens seen so far:
       step 1: 5    step 2: 6    step 3: 7    step 4: 8     total = 26
    With a cache, only the new token is computed after the prefill:
       step 1: 5    step 2: 1    step 3: 1    step 4: 1     total = 8
    The same numbers hold for V, and they repeat in every layer and every head.

(b) computed in this step : the q of token 6, the k and v of token 6
    read from the cache   : the k and v of tokens 1-5
    not needed at all     : the q of tokens 1-5 (their outputs were already
                            computed in step 1)

(c) 8 rows: the 5 prompt tokens + tokens 6, 7 and 8, so each K cache is 8 × 64.
    One K matrix and one V matrix per layer and per head:
       4 layers × 8 heads = 32 K matrices and 32 V matrices

(d) one token    : 2 × 4 × 8 × 64 × 2 = 8,192 bytes = 8 KiB
    1,024 tokens : 8 KiB × 1,024 = 8,192 KiB = 8 MiB

(e) GQA-2: one token = 2 × 4 × 2 × 64 × 2 = 2,048 bytes = 2 KiB  →  2 MiB per sequence
    MQA  : one token = 2 × 4 × 1 × 64 × 2 = 1,024 bytes = 1 KiB  →  1 MiB per sequence
    Only n_kv_heads changes (8 → 2 → 1), so the cache is 4× and 8× smaller than with MHA.

(f) MHA  : 64 MiB ÷ 8 MiB = 8 sequences
    GQA-2: 64 MiB ÷ 2 MiB = 32 sequences
    MQA  : 64 MiB ÷ 1 MiB = 64 sequences

(g) 1. False. The q of an old token is only used to compute that token's own
       output, which was computed in an earlier step and is never needed again.
    2. False. The KV cache is exact: the output is the same as without it.
    3. True. No token can attend to the tokens on its right, so appending a
       token cannot change the earlier ones.
    4. False. In pre-training (and fine-tuning) the whole text is known in advance
       and processed in one parallel pass, so nothing is cached.
    5. True. GQA saves memory, not attention computation.
    6. False. The encoder reads the whole input only once, in one parallel pass.
       Only the decoder, which runs once for every generated token, needs a cache.
```
</details>

<br><br><br><br><br><br><br><br><br><br><br><br><br><br>
# <span style='background :lightgreen' >13. Mixture of Experts (MoE)</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">A <b>Mixture of Experts (MoE)</b> layer replaces the single feed-forward network of a transformer layer with several feed-forward networks called <b>experts</b>, plus a small <b>router</b> that sends each token to only a few of them. The model <b>stores</b> many parameters but <b>uses</b> only a small part of them for each token.</div></h3>

## a. The Idea: Many Experts, Few Used per Token
- In a **dense** model (everything up to §12) every token passes through **all** the weights of every layer. A bigger model therefore means more computation for every single token.
- MoE breaks this link. Each MoE layer contains **N experts** (e.g., 8), but a small network called the **router** picks only **k** of them (e.g., 2) for each token. The other experts are not run for that token at all.
- This is called **sparse activation** or **conditional computation**: the model can store far more parameters (more knowledge), while the computation per token stays close to that of a much smaller dense model.
- **Why do we want this?** Larger models know more and perform better, but in a dense model every extra parameter costs extra computation for every token. MoE lets the number of parameters grow much faster than the computation per token. Mixtral 8×7B, for example, stores 46.7B parameters but uses only 12.9B for each token, and it matches or beats the dense Llama-2-70B on most benchmarks.
- **An analogy:** a hospital has 8 specialists and a receptionist. The receptionist (the router) sends each patient (a token) to 2 of the specialists (the experts). The hospital must pay all 8 specialists (the memory), but each patient takes the time of only 2 (the computation). Unlike real doctors, the experts do not specialize in human topics (§13d).

<div style="text-align:center;">
    <img src="../images/moe-05.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

- A short history: the idea goes back to *Adaptive Mixtures of Local Experts* (Jacobs, Jordan, Nowlan and Hinton, 1991). Shazeer et al. (2017) brought it to deep learning with the **sparsely-gated MoE layer** and its **noisy top-k gating**, which we follow below. GShard (2020) and the Switch Transformer (2021) scaled it up inside transformers, and today many of the largest open models are MoE models (table in §13h).

<br><br><br><br>

## b. Where the Experts Live

<div style="text-align:center;">
    <img src="../images/moe-01.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

- An MoE layer **replaces the FFN** of §8. Masked self-attention, the norms and the residual connections stay exactly the same.
- Each expert is an ordinary FFN with its own weights, usually with the same shape as the dense FFN; in our running example 512 → 2048 → 512.
- **Why replace the FFN and not attention?** Attention has 4 × d_model² parameters (W_Q, W_K, W_V and W_O), while the FFN has 2 × d_model × 4·d_model = 8 × d_model², so the FFN holds about **two thirds** of the parameters of a layer. The FFN also processes every token on its own, without looking at the other tokens, so each token can easily be sent to a different FFN.
- Routing happens **per token and per layer**: the 8 tokens of *"Ask not what your country can do for"* can each go to different experts, and the same token can use different experts in different layers.
- Many MoE models (e.g., Mixtral) use an MoE layer in every transformer layer. Others replace only every second FFN (GShard) or keep the first few layers dense (DeepSeek-V3 keeps its first 3 layers dense).

<br><br><br><br>

## c. The Router: Noisy Top-k Gating

<div style="text-align:center;">
    <img src="../images/moe-02.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

The router is one linear layer with a learned weight matrix $W_g$ of shape (d_model × N), trained together with the experts. For one token vector $x$ (1 × d_model), the MoE layer works in five steps (Shazeer et al., 2017):

- **① Router scores:** $H(x) = x\,W_g$, one score per expert (shape 1 × N). A higher score means the router thinks this expert suits the token better.
- **② Noise (training only):** $H(x)_i = (x\,W_g)_i + \text{StandardNormal}() \cdot \text{Softplus}\big((x\,W_{noise})_i\big)$. The random noise makes the router try different experts during training, which helps to spread the tokens over all experts (§13f). It is switched off at inference, so *"ignore the noise step"* simply means $H(x) = x\,W_g$.
- **③ KeepTopK:** keep the k largest scores and set all the others to $-\infty$:

$$\text{KeepTopK}(v, k)_i = \begin{cases} v_i & \text{if } v_i \text{ is among the } k \text{ largest elements of } v \\ -\infty & \text{otherwise} \end{cases}$$

- **④ Softmax:** $G(x) = \text{Softmax}\big(\text{KeepTopK}(H(x), k)\big)$. Because $e^{-\infty} = 0$, every expert that was not selected gets a gate of **exactly 0**, and the k selected gates sum to 1. This is the same $-\infty$ trick as the causal mask in §5.
- **⑤ Weighted sum:** $y = \sum_{i=1}^{N} G(x)_i \, E_i(x)$, where $E_i(x)$ is the output of expert i. Every term with $G(x)_i = 0$ adds nothing, so those experts are simply **not run**: only k expert FFNs are computed for this token.
- **Worked example (figure above):** the scores are [0.5, 2.4, −0.3, 1.3, 0.8, −1.2, 0.1, 1.0] and k = 2.
```
KeepTopK keeps E2 (2.4) and E4 (1.3):   [−∞, 2.4, −∞, 1.3, −∞, −∞, −∞, −∞]
Softmax:  e^2.4 = 11.023,  e^1.3 = 3.669,  sum = 14.692
          g2 = 11.023 / 14.692 = 0.750     g4 = 3.669 / 14.692 = 0.250     all other gates = 0
Output:   y = 0.750 × E2(x) + 0.250 × E4(x)        ← only E2 and E4 are computed
```
- **Variants you will meet:**
    - **Mixtral 8×7B** uses exactly this rule with N = 8 and k = 2 (softmax after the top-k selection).
    - The **Switch Transformer** uses k = 1. It applies the softmax to **all** N scores first, sends the token to the single best expert, and scales that expert's output by its probability, so the gate is less than 1.
    - **DeepSeek-V3** uses sigmoid scores instead of a softmax and normalizes them over the selected experts.
- **How does the router learn?** $W_g$ is trained together with the rest of the model by backpropagation. The gates multiply the expert outputs (step ⑤), so the training loss tells the router whether a larger or a smaller gate would have helped. Over time it learns to give higher scores to the experts that work well for each kind of token.

<br><br><br><br>

## d. Total vs Active Parameters

<div style="text-align:center;">
    <img src="../images/moe-03.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

- Every token picks its own k experts, so across a batch almost all experts are used, but each single token touches only k of the N.
- An MoE model therefore has two parameter counts:
    - **Total parameters:** everything that must be **stored** in GPU memory (all experts of all layers).
    - **Active parameters:** what is **used** to process **one** token (the shared parts plus k experts per layer).
- **The computation per token is proportional to the active parameters** (about one multiplication and one addition per active parameter). So with **top-1** routing and experts of the same size as the dense FFN, an MoE layer needs **exactly the same computation** as the dense FFN, while it stores N times more FFN parameters. With top-2 it needs twice the FFN computation.
- **Example: Mixtral 8×7B** (32 layers, d_model = 4096, 8 experts per layer, top-2; each expert is a SwiGLU FFN with three 4096 × 14,336 matrices):
```
one expert          = 3 × 4,096 × 14,336                    = 176,160,768 parameters
all experts         = 32 layers × 8 × 176,160,768            = 45.10 B
shared parameters   = attention 1.34 B + embeddings and LM head 0.26 B + routers 0.001 B   ≈ 1.61 B
total  (stored)     = 45.10 B + 1.61 B                       ≈ 46.7 B
active (per token)  = 32 × 2 × 176,160,768 + 1.61 B          = 11.27 B + 1.61 B ≈ 12.9 B
```
- The name "8×7B" is misleading: the model is **not** 8 × 7 = 56B, because only the FFNs are copied 8 times. Attention, embeddings and the LM head are shared.
- So Mixtral **computes** like a 12.9B model but needs the **memory** of a 46.7B model: 46.7B × 2 bytes ≈ 93 GB in 16-bit, which does not fit on one 80 GB GPU.
- **Why not load only the active experts?** The router chooses the experts at run time, separately for every token in every layer, and the tokens of a batch go to different experts. So all the experts must be ready in GPU memory. (Some local tools keep the experts in CPU memory and copy them to the GPU when they are needed: this needs less GPU memory, but it is much slower.)
- **MoE does not change the KV cache.** Only the FFN is replaced; attention is untouched. Mixtral has 32 layers, 8 KV heads and d_head = 128, so its cache is 2 × 32 × 8 × 128 × 2 bytes = 128 KiB per token, exactly like Llama-3.1-8B in §12h.
- MoE and the techniques of §12 solve **different** problems:

| Technique | What it saves | What it does **not** save |
|---|---|---|
| **GQA / MQA** (§12i) | KV-cache memory and memory bandwidth | attention computation (still O(n²)) |
| **MoE** (§13) | FFN computation per token | memory: every expert must be stored |

- **Do experts specialize in topics?** The name suggests a "biology expert" or a "maths expert", but that is not what happens. The Mixtral paper found no clear topic specialization; the routing followed more syntactic patterns, for example consecutive tokens are often sent to the same expert.

<br><br><br><br>

## e. Build It: an MoE Layer in NumPy
- The code below builds one MoE layer with 8 tiny experts and top-2 routing, exactly as in the five steps of §13c, and routes the 8 tokens of our sentence through it.
- For every token it checks that running **only the 2 selected experts** gives the same output as running all 8 experts and weighting them by the gates. It also counts how many tokens each expert receives.

In [2]:
import numpy as np

np.random.seed(0)
d_model, d_ff, n_experts, k = 16, 64, 8, 2              # tiny sizes so that the demo runs instantly

# ── 8 experts: each one is an ordinary FFN (d_model → d_ff → d_model) with its own weights ──
experts = [(np.random.randn(d_model, d_ff) * 0.1, np.random.randn(d_ff, d_model) * 0.1) for _ in range(n_experts)]
W_g = np.random.randn(d_model, n_experts)                # router weights (d_model × n_experts)

def ffn(x, W1, W2):
    return np.maximum(0, x @ W1) @ W2                     # ReLU FFN as in §8 (biases left out)

def softmax(v):
    e = np.exp(v - v.max())
    return e / e.sum()

def moe_layer(x):
    scores = x @ W_g                                      # ① router scores H(x), one per expert
    top = np.argsort(scores)[::-1][:k]                    # indices of the k largest scores
    masked = np.full(n_experts, -np.inf)
    masked[top] = scores[top]                             # ③ KeepTopK: all other scores become −∞
    gates = softmax(masked)                               # ④ softmax: unselected experts get exactly 0
    y = sum(gates[i] * ffn(x, *experts[i]) for i in top)  # ⑤ run ONLY the k selected experts
    return y, top, gates

tokens = ["Ask", "not", "what", "your", "country", "can", "do", "for"]
X = np.random.randn(len(tokens), d_model)                # stand-in vectors for the 8 tokens
counts = np.zeros(n_experts, dtype=int)
print(f"{'token':<9}{'experts':<10}{'gates':<15}{'same as running all 8 experts?':>32}")
for tok, x in zip(tokens, X):
    y, top, gates = moe_layer(x)
    y_all = sum(gates[i] * ffn(x, *experts[i]) for i in range(n_experts))   # run all 8, weight by the gates
    counts[top] += 1
    print(f"{tok:<9}{', '.join(f'E{i+1}' for i in top):<10}{', '.join(f'{gates[i]:.3f}' for i in top):<15}{str(np.allclose(y, y_all)):>32}")

print("\ntokens per expert:", "  ".join(f"E{i+1}={c}" for i, c in enumerate(counts)))
per_expert = d_model * d_ff + d_ff * d_model              # parameters of one expert (W1 and W2)
print(f"expert parameters stored : {n_experts} × {per_expert:,} = {n_experts * per_expert:,}")
print(f"expert parameters used   : {k} × {per_expert:,} = {k * per_expert:,} per token")

token    experts   gates            same as running all 8 experts?
Ask      E2, E7    0.922, 0.078                               True
not      E4, E2    0.956, 0.044                               True
what     E4, E6    0.853, 0.147                               True
your     E6, E5    0.726, 0.274                               True
country  E6, E8    0.780, 0.220                               True
can      E1, E7    0.794, 0.206                               True
do       E3, E1    0.800, 0.200                               True
for      E3, E6    0.860, 0.140                               True

tokens per expert: E1=2  E2=2  E3=2  E4=2  E5=1  E6=4  E7=2  E8=1
expert parameters stored : 8 × 2,048 = 16,384
expert parameters used   : 2 × 2,048 = 4,096 per token


- **Same output for every token:** skipping the unselected experts is exact, not an approximation, because their gates are exactly 0. MoE saves this work for free.
- **Work saved:** each token uses 2 × 2,048 = 4,096 expert parameters instead of 16,384. In a real model the experts are far larger, and the same saving happens in every MoE layer.
- **Uneven load:** E6 received 4 of the 16 token-to-expert assignments, while E5 and E8 received only 1 each. In a large batch this imbalance becomes a real problem, which is the topic of the next part.

<br><br><br><br>

## f. Load Balancing: Keep Every Expert Busy

<div style="text-align:center;">
    <img src="../images/moe-04.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

- **Router collapse:** early in training some experts happen to be slightly better, so the router sends them more tokens, so they learn faster and become even better, and so on. The other experts receive almost no tokens and barely learn. The model then wastes most of its experts.
- **Expert capacity:** in training, experts process tokens in fixed-size batches, so each expert can take at most

$$\text{expert capacity} = \text{capacity factor} \times \frac{k \times \text{tokens in the batch}}{N}$$

- With 1,024 tokens, N = 8, top-1 routing (k = 1, as in the Switch Transformer) and a capacity factor of 1.25, each expert takes at most 1.25 × 1 × 1,024 / 8 = **160** tokens. (With top-2, every token makes 2 assignments, so the capacity doubles.)
- Tokens that arrive at a full expert **overflow** and are **dropped** for this layer: they skip the expert and continue unchanged through the residual connection.
- A capacity factor above 1 leaves room for some imbalance; a very large one wastes memory and computation.

**Fixes for an unbalanced router:**
- **Noise** in the router scores (step ② of §13c) makes the router try other experts.
- **Auxiliary load-balancing loss** (Switch Transformer), added to the normal training loss:

$$L_{aux} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i \cdot P_i$$


  where $f_i$ is the fraction of tokens sent to expert i, $P_i$ is the average router probability for expert i over the batch, and α is a small weight (0.01 in the Switch Transformer). The loss is smallest when every expert gets the same share 1/N.

- **Auxiliary-loss-free balancing** (DeepSeek-V3): a bias is added to each expert's score, used only to choose the top-k experts. After each training step the bias of an overloaded expert is lowered and the bias of an underused expert is raised.

<br><br><br><br>

## g. MoE in Practice: Batches, GPUs and Speed

<div style="text-align:center;">
    <img src="../images/moe-06.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

- **Fast decoding for one user:** a decode step is memory-bandwidth-bound (§12e): its time depends on how many weights must be read from GPU memory. For a single sequence, an MoE model reads only the experts it uses, so Mixtral generates tokens at about the speed of a 13B dense model, not of a 47B one. With large batches the tokens together use almost all the experts, so almost all the weights are read in every step.
- **The costs of MoE:** all the experts must fit in memory (§13d), training needs load balancing (§13f), the GPUs must exchange tokens (figure above), and MoE models are harder to fine-tune: with little data the experts tend to overfit.

<br><br><br><br>

## h. MoE Models You May Know

| Model | Year | Experts per MoE layer | Experts used per token | Total parameters | Active parameters |
|---|---|---|---|---|---|
| Mixtral 8×7B (Mistral AI) | 2023 | 8 | 2 | 46.7 B | 12.9 B |
| DeepSeek-V3 (DeepSeek) | 2024 | 256 routed + 1 shared | 8 routed + 1 shared | 671 B | 37 B |
| Qwen3-235B-A22B (Alibaba) | 2025 | 128 | 8 | 235 B | 22 B |
| gpt-oss-120b (OpenAI) | 2025 | 128 | 4 | 117 B | 5.1 B |
| gpt-oss-20b (OpenAI) | 2025 | 32 | 4 | 21 B | 3.6 B |

- Model names often give the active size away: *A22B* in Qwen3-235B-A22B means **22B active** parameters.
- The gpt-oss models that are available on Groq are MoE models too: gpt-oss-20b stores 21B parameters but uses only about 3.6B for each token.
- **Fine-grained experts:** modern MoE models use many small experts instead of a few large ones. Cutting every expert into 4 and choosing 4 times as many per token keeps the parameters and the computation the same, but gives far more possible combinations of experts for a token, so the experts can specialize more. The number of combinations is C(N, k) = N! / (k! (N − k)!), e.g., C(16, 2) = 120.
- **Shared experts** (DeepSeek): one expert is used by **every** token to hold common knowledge, so that the routed experts do not all have to learn it again.

<div style="text-align:center;">
    <img src="../images/moe-07.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

## **Q10. MoE Routing with KeepTopK**

An MoE layer has **4 experts**. For one token the router produces the scores **[2.0, 1.0, 0.5, −1.0]** (experts 1 to 4). The layer uses **KeepTopK with k = 2**. Ignore the noise step and answer the following questions:

**(a)** Write the score vector after KeepTopK masking.

**(b)** Compute the gating weights after softmax, to three decimal places.

**(c)** Expert 1 outputs [1, 2] and expert 2 outputs [3, 0] for this token. Compute the MoE layer's output. Why are experts 3 and 4 never run?

**(d)** Why does KeepTopK set the unselected scores to **−∞** and not to **0**? Use the same scores [2.0, 1.0, 0.5, −1.0] to show what would happen with 0.

**(e)** The **Switch Transformer** uses k = 1, and it applies the softmax to **all** scores before choosing the best expert. For the same token, which expert runs, what is its gate, and what is the output? Compare it with (c).

<details>
<summary>💡 <b>Click to reveal answer</b></summary>

```
Given:  scores H(x) = [2.0, 1.0, 0.5, −1.0],  k = 2,  no noise

(a) KeepTopK keeps the 2 largest scores and sets the rest to −∞:
      the 2 largest are 2.0 (expert 1) and 1.0 (expert 2)

      KeepTopK(H(x), 2) = [2.0, 1.0, −∞, −∞]

(b) Softmax of [2.0, 1.0, −∞, −∞]:
      e^2.0  = 7.389
      e^1.0  = 2.718
      e^(−∞) = 0          (for experts 3 and 4)
      sum    = 7.389 + 2.718 + 0 + 0 = 10.107

      g1 = 7.389 / 10.107 = 0.731
      g2 = 2.718 / 10.107 = 0.269
      g3 = 0 / 10.107     = 0
      g4 = 0 / 10.107     = 0

      G(x) = [0.731, 0.269, 0, 0]
      Check: 0.731 + 0.269 + 0 + 0 = 1.000

(c) y = Σ G(x)_i × E_i(x)  →  only experts 1 and 2 have a non-zero gate:

      y = 0.731 × [1, 2] + 0.269 × [3, 0]
        = [0.731, 1.462] + [0.807, 0.000]
        = [1.538, 1.462]

    Why experts 3 and 4 are never run (for this token):
    Their gates are EXACTLY 0, so their terms 0 × E3(x) and
    0 × E4(x) would be 0 whatever the experts output. The layer
    therefore does not compute E3 and E4 at all. This is the whole
    point of a sparse MoE: only k = 2 of the 4 expert FFNs do any
    work, which halves the FFN computation for this token.
    The gates are exactly 0 (not just small) because KeepTopK used
    −∞ and e^(−∞) = 0. Other tokens may still be routed to experts
    3 and 4, so those experts must stay in memory.

(d) With −∞, e^(−∞) = 0, so the unselected experts get a gate of EXACTLY 0
    and do not need to be run (see (b): gates [0.731, 0.269, 0, 0]).

    With 0 instead, softmax([2.0, 1.0, 0, 0]):
      e^2.0 = 7.38906,  e^1.0 = 2.71828,  e^0 = 1,  e^0 = 1
      sum   = 12.10734
      gates = [0.6103, 0.2245, 0.0826, 0.0826]   (sum = 1.0000)

    Experts 3 and 4 would still receive about 8% weight each, so they
    would have to be run too: no computation is saved and the layer is
    no longer "top-2". (The same reason the causal mask in §5 uses −∞.)

(e) Softmax over ALL four scores [2.0, 1.0, 0.5, −1.0]:
      e^2.0 = 7.38906,  e^1.0 = 2.71828,  e^0.5 = 1.64872,  e^(−1.0) = 0.36788
      sum   = 12.12394
      p     = [0.6095, 0.2242, 0.1360, 0.0303]

    Top-1 → only expert 1 runs, with gate p1 = 0.6095 (NOT renormalized to 1):
      y = p1 × [1, 2] = [0.6095, 1.2189]

    Compared with (c) (top-2, y = [1.538, 1.462]):
      - only ONE expert FFN is computed instead of two (half the expert compute)
      - the gate is below 1, because the softmax also gave probability to
        the experts that were not chosen
```
</details>

## **Q11. MoE Concepts**

**(a)** A student says: *"An MoE model with 8 experts and top-2 routing uses only 2 of its 8 experts per token, so it needs only 2/8 = 25% of the GPU memory for its experts."* Is the student right? Explain.

**(b)** During training, the router of an 8-expert MoE layer sends 90% of all tokens to expert 1. What is this problem called, why is it harmful, and name two ways to fix it.

**(c)** A Switch Transformer layer has 8 experts and receives a batch of 1,024 tokens (top-1 routing) with a capacity factor of 1.25. What is the expert capacity? The router sends 200 tokens to expert 3. What happens to them?

**(d)** Mixtral 8×7B is serving a single user (batch size 1). Why does it generate tokens about as fast as a 13B dense model, although it needs the memory of a 47B model?

**(e)** A server runs out of GPU memory because of very long contexts. Would converting the model to MoE help? Which technique from §12 would help instead?

**(f)** An MoE layer has 8 experts and uses top-2. Every expert is now cut into 2 halves, giving 16 experts, and the layer uses top-4. Does the computation per token change? How many different combinations of experts can a token use before and after the change?

<details>
<summary>💡 <b>Click to reveal answer</b></summary>

```
(a) The student is WRONG.
    Top-2 means each TOKEN uses only 2 of the 8 experts, which saves
    COMPUTATION per token. But different tokens choose different experts
    (in every layer), so all 8 experts must be stored in GPU memory.
    The memory is the same as storing all experts; only the compute
    per token shrinks.

(b) This is ROUTER COLLAPSE (an unbalanced load).
    It is harmful because:
      - the other 7 experts receive almost no tokens, so they barely learn:
        their parameters are stored but wasted
      - expert 1 overflows its capacity, so many of its tokens are dropped
    Fixes (any two):
      - add noise to the router scores during training (noisy top-k gating)
      - an auxiliary load-balancing loss  L_aux = α · N · Σ f_i · P_i
      - auxiliary-loss-free balancing with per-expert biases (DeepSeek-V3)

(c) Expert capacity = capacity factor × k × tokens / N
                    = 1.25 × 1 × 1,024 / 8
                    = 160 tokens per expert

    Expert 3 can process only 160 of its 200 tokens.
    The other 200 − 160 = 40 tokens OVERFLOW and are DROPPED for this
    layer: they skip the expert and pass through unchanged via the
    residual connection. (They are still processed by the other layers.)

(d) A decode step is memory-bandwidth-bound: its time depends on how many
    weights are READ from GPU memory. With one sequence, each token reads
    only the shared parts and 2 of the 8 experts per layer, i.e. about
    12.9B parameters, like a 13B dense model.
    But all 46.7B parameters must be STORED, because the next token (and
    every layer) may choose different experts.

(e) NO. MoE saves FFN COMPUTATION, not memory, and it does not change
    the KV cache, which is what grows with long contexts. It would even
    add memory, because all the experts must be stored.
    The §12 techniques that help are the ones that shrink the KV cache:
    GQA or MQA (fewer K, V heads), MLA (compressed cache), quantizing the
    cache, or sliding windows / eviction (fewer stored tokens).

(f) Computation: before, 2 experts of size 1 = 2 units.
                 after,  4 experts of size ½ = 2 units  → NO change.
    (The stored parameters do not change either: 16 × ½ = 8 × 1.)
    Combinations: before C(8, 2)  = 8 × 7 / 2               = 28
                  after  C(16, 4) = 16 × 15 × 14 × 13 / 24  = 1,820
    65 times more combinations for the same computation: this is the
    idea of fine-grained experts.
```
</details>

## **Q12. Total vs Active Parameters**

A small decoder-only model has **4 layers** and **d_model = 100**. Every layer has:
- multi-head attention with four weight matrices W_Q, W_K, W_V and W_O, each of shape 100 × 100 (4 heads with d_head = 25)
- an MoE layer with **8 experts** and **top-2** routing; each expert is a ReLU FFN with W1 (100 × 400) and W2 (400 × 100)

The embedding matrix (10,000 × 100) is shared with the LM head. Ignore biases, norms and the routers.

**(a)** How many parameters does one expert have?

**(b)** How many parameters does the model store in total?

**(c)** How many parameters are active for one token? What fraction of the total is that?

**(d)** How much memory do the weights need in 16-bit? Could the GPU load only the active experts to save memory?

**(e)** How many parameters would the same model store with one dense FFN (100 → 400 → 100) per layer instead of the MoE layer? In one layer, compare the FFN computation per token of the dense model, the MoE model with top-1 and the MoE model with top-2.

**(f)** Does replacing the dense FFN by the MoE layer change the KV cache? Compute the KV cache per token (16-bit numbers).

<details>
<summary>💡 <b>Click to reveal answer</b></summary>

```
(a) One expert = W1 + W2
               = 100 × 400 + 400 × 100
               = 40,000 + 40,000
               = 80,000 parameters

(b) Per layer:
      attention = 4 × 100 × 100    = 40,000
      experts   = 8 × 80,000       = 640,000
      one layer = 40,000 + 640,000 = 680,000
    All 4 layers = 4 × 680,000     = 2,720,000
    Embeddings (shared with LM head) = 10,000 × 100 = 1,000,000

    Total = 2,720,000 + 1,000,000 = 3,720,000  ≈ 3.7 M parameters

(c) Per layer, a token uses attention + only 2 experts:
      40,000 + 2 × 80,000 = 200,000
    All 4 layers = 4 × 200,000 = 800,000
    Active = 800,000 + 1,000,000 = 1,800,000  = 1.8 M parameters

    Fraction = 1,800,000 / 3,720,000 ≈ 0.48, i.e. about 48% of the total

(d) Memory = 3,720,000 × 2 bytes = 7,440,000 bytes ≈ 7.4 MB

    NO, the GPU cannot load only the active experts. The router chooses
    the experts separately for EVERY token in EVERY layer, and the tokens
    of a batch go to different experts, so all 8 experts of every layer
    must be in memory. MoE saves computation, not memory.

(e) Dense model, one FFN per layer:
      one layer = 40,000 + 80,000 = 120,000
      total     = 4 × 120,000 + 1,000,000 = 1,480,000  ≈ 1.5 M

    FFN parameters used per token in one layer (∝ FFN computation):
      dense FFN        : 80,000
      MoE, top-1       : 1 × 80,000 = 80,000   → the SAME computation as dense
      MoE, top-2       : 2 × 80,000 = 160,000  → 2× the dense computation
    But the MoE layer STORES 8 × 80,000 = 640,000 FFN parameters,
    8× more than the dense FFN (more capacity to learn).

(f) NO. MoE replaces only the FFN; attention, and so the KV cache,
    is unchanged:
      KV cache per token = 2 × n_layers × n_kv_heads × d_head × bytes
                         = 2 × 4 × 4 × 25 × 2
                         = 1,600 bytes   (dense and MoE alike)
```
</details>